# 08 published architectures under the flight-level protocol, and real-log missingness

Design decisions:
- Published architectures (E7). Two detectors from the literature are re-implemented and run under exactly the protocol of this paper: the hybrid LSTM-GRU sequence model of reference [6] (LSTM 64 units followed by GRU 32 units on sequences of six consecutive window feature vectors, class-weighted cross-entropy, six epochs, no tuning) and the one-class novelty detector of reference [5] (isolation forest trained on nominal windows only, threshold at the 95th percentile of nominal training scores, nominal versus attack). Both get the paired leakage audit on the same splits as E0, the in-distribution flight-level evaluation on the same balanced test sets as E1 (the sequence model through the same probability-summary stacking; the one-class detector through a declared flight rule: flagged when more than 10 percent of windows are flagged), and the unseen-subtype test with the same single-fit allocation as E2 (one seed).
- Missingness on real flights (E6 rerun). For each real flight the fraction of missing base features is recorded and the verdict, abstention and gate rates are split at 5 percent missing.
- Onset criteria (E5 rerun). The within-30 s criterion is added beside within-10 s and within-60 s; results are otherwise identical.
- The first two code cells write the updated `src/sih_model.py` and `src/sih_report.py` into the repository so that nothing is copied by hand.

Result files: `reports/v4/e7_baselines_leakage.csv`, `e7_baselines_flight.csv`, `e7_baselines_loso.csv`, updated `e6_real_summary.csv` and `e6_real_flight_verdicts.csv` (with `missing_frac`), updated `e5_onset_summary.csv`. Manuscript: Sections 6.1, 6.5 and 6.7, Tables 10, 14, 16.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
%%writefile /content/drive/MyDrive/UAV_GNSS_Research/uav-gnss-triage/src/sih_model.py
#!/usr/bin/env python3
"""
sih_model.py  --  window detector + stacked flight classifier + trust layer, evaluated at the FLIGHT level.

Feature engineering on top of windows.csv (all per flight, so every split stays flight-grouped):
  * temporal context: first differences and rolling maxima over 3 windows for the core physics features
  * flight-relative features: sensor-health and consistency features divided by the flight's own median over
    its first 20 s (an investigator compares a recovered log against itself; also absorbs receiver/airframe offsets)

Flight verdict: window probabilities are produced out-of-fold on the training flights (4-fold by flight),
aggregated per flight (max / p90 / mean / fraction > 0.5 / longest run per class, plus raw maxima of gap fractions
and consistency residuals), and a regularised multinomial logistic model is fitted on those aggregates.
The percentile rule from the first version is kept as a comparison ("rule" vs "stacked").

Experiments
  E0  Leakage audit, paired: per repetition, a flight-grouped split (30 training flights per family) and a random-window
      split with the same number of training windows, same detector; per-rep difference reported.
  E1  In-distribution, flight-grouped: 20 train / 20 calibration / 20 test flights per family (balanced test so the
      within-coarse-class mixture matches calibration), R repetitions; extra flights beyond 20 per family are scored
      separately. Window metrics raw vs temperature-scaled; flight-level accuracy, macro-F1, AURC and probability
      calibration (ECE, NLL, Brier) for the rule and for three stacked variants (probability summaries only, raw flight
      features only, full); a sensitivity variant without the derived-topic gap features; class-conditional conformal sets
      at the flight level (alpha 0.10 and 0.20) with abstention decomposed into multi-label, empty, non-singleton and
      operational (non-singleton or verdict outside its set).
  E2  Leave-one-subtype-out with the full stacked pipeline, with the split allocation and conformal ranks exported and a
      matched-size seen-subtype control for every hold-out.
  E3  Whelan live logs as case studies: verdicts, sets, window timeline, receiver-derived disturbance intervals and a
      declared alert rule with detection delay and false alerts.

Rules: receiver self-report (rx_*) columns are excluded from the detector. Simulator truth is never used.
Usage:
  python sih_model.py --features <features dir> --out <reports dir>
"""
import argparse
import os
import json
import pathlib
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
CLASSES = ['nominal', 'spoof', 'gps_degrade', 'sensor_fault']
META = {'t_start', 't_end', 'flight_id', 'source', 'run', 'family', 'subtype', 'onset_s', 'restore_s',
        'home_lat', 'noise_k', 'noise_t', 'noise_p', 'window_label', 'log'}
COARSE = {'nominal': 'nominal', 'spoof': 'gnss_chain', 'gps_degrade': 'gnss_chain', 'sensor_fault': 'non_gnss_fault'}
COARSE_CLASSES = ['nominal', 'gnss_chain', 'non_gnss_fault']
CTX_FEATS = ['posvel_rms', 'step_minus_vel_max', 'step_speed_max', 'gps_baro_dz_rms', 'dv_gps_imu_vecdiff',
             'gps_ekf_dpos_rms', 'gps_gap_frac', 'gps_nofix_frac', 'gps_silent_frac', 'baro_gap_frac', 'mag_gap_frac',
             'baro_raw_std', 'mag_raw_std',
             'tr_gpsHpos0_absmax', 'tr_gpsHpos1_absmax', 'tr_gpsHvel0_absmax', 'tr_gpsVpos_absmax',
             'tr_baroVpos_absmax', 'tr_heading_absmax', 'inn_gpsHpos0_absmax', 'inn_gpsHvel0_absmax']
REL_FEATS = ['baro_raw_std', 'mag_raw_std', 'mag_norm_std', 'mag_norm_mean', 'gyro_std_mean', 'baro_std',
             'posvel_rms', 'gps_ekf_dpos_rms', 'gps_baro_dz_rms', 'acc_horiz_rms', 'inn_gpsHpos0_absmax',
             'inn_gpsHvel0_absmax', 'inn_baroVpos_absmax', 'inn_heading_absmax']
AGG_RAW = ['gps_gap_frac', 'gps_nofix_frac', 'gps_silent_frac', 'baro_gap_frac', 'mag_gap_frac', 'posvel_rms',
           'step_minus_vel_max', 'gps_ekf_dpos_rms',
           'gps_baro_dz_rms', 'tr_gpsHpos0_absmax', 'tr_baroVpos_absmax', 'tr_heading_absmax']


# ------------------------------------------------------------------ feature engineering (per flight)
def engineer(W):
    W = W.sort_values(['flight_id', 't_start']).reset_index(drop=True)
    g = W.groupby('flight_id', sort=False)
    new = {}
    for f in [f for f in CTX_FEATS if f in W.columns]:
        new[f + '_d1'] = g[f].diff()
        new[f + '_rmax3'] = g[f].transform(lambda s: s.rolling(3, min_periods=1).max())
    rel = [f for f in REL_FEATS if f in W.columns]
    early = W[W['t_start'] < 20.0].groupby('flight_id')[rel].median()
    for f in rel:
        base = W['flight_id'].map(early[f]).astype(float)
        new[f + '_rel'] = W[f] / (base.abs() + 1e-3)
    return pd.concat([W, pd.DataFrame(new, index=W.index)], axis=1)


# ------------------------------------------------------------------ models
def make_xgb(seed):
    try:
        from xgboost import XGBClassifier
        return XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, tree_method='hist', objective='multi:softprob',
                             random_state=seed, n_jobs=-1, verbosity=0)
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_depth=5, random_state=seed)


def make_logreg(seed):
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                         LogisticRegression(max_iter=3000, C=1.0, class_weight='balanced', random_state=seed))


def make_flight_model(seed):
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                         LogisticRegression(max_iter=5000, C=0.3, class_weight='balanced', random_state=seed))


def fit(model, X, y):
    yi = np.array([CLASSES.index(v) for v in y])
    counts = np.bincount(yi, minlength=len(CLASSES)).astype(float)
    w = (len(yi) / (len(CLASSES) * np.maximum(counts, 1)))[yi]
    try:
        model.fit(X, yi, sample_weight=w)
    except (TypeError, ValueError):
        model.fit(X, yi)
    return model


def predict_proba(model, X):
    p = model.predict_proba(X)
    full = np.full((len(X), len(CLASSES)), 1e-6)
    cls = getattr(model, 'classes_', None)
    if cls is None:
        cls = model[-1].classes_
    for j, c in enumerate(cls):
        full[:, int(c)] = p[:, j]
    return full / full.sum(1, keepdims=True)


# ------------------------------------------------------------------ calibration metrics
def ece_top(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)


def nll(p, y):
    return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())


def brier(p, y):
    oh = np.eye(p.shape[1])[y]
    return float(((p - oh) ** 2).sum(1).mean())


def fit_temperature(p_cal, y_cal):
    logit = np.log(np.clip(p_cal, 1e-12, 1))
    def loss(T):
        z = logit / T
        z = z - z.max(1, keepdims=True)
        q = np.exp(z); q /= q.sum(1, keepdims=True)
        return nll(q, y_cal)
    r = minimize_scalar(loss, bounds=(0.05, 20.0), method='bounded')
    return float(r.x)


def apply_temperature(p, T):
    z = np.log(np.clip(p, 1e-12, 1)) / T
    z = z - z.max(1, keepdims=True)
    q = np.exp(z)
    return q / q.sum(1, keepdims=True)


# ------------------------------------------------------------------ flight-level aggregation
def longest_run(mask):
    best = cur = 0
    for v in mask:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best


def flight_aggregates(W, P):
    """Aggregate window probabilities (and a few raw window features) into one row per flight."""
    rows = []
    raw = [f for f in AGG_RAW if f in W.columns]
    for fid, idx in W.groupby('flight_id', sort=False).indices.items():
        p = P[idx]; w = W.iloc[idx]
        r = {'flight_id': fid}
        for i, c in enumerate(CLASSES):
            col = p[:, i]
            r[f'max_{c}'] = float(col.max()); r[f'p90_{c}'] = float(np.percentile(col, 90))
            r[f'mean_{c}'] = float(col.mean()); r[f'frac_{c}'] = float((col > 0.5).mean())
            r[f'run_{c}'] = float(longest_run(col > 0.5))
        for f in raw:
            r[f'{f}_max'] = float(np.nanmax(w[f])) if np.isfinite(w[f]).any() else np.nan
        rows.append(r)
    A = pd.DataFrame(rows)
    fam = W.groupby('flight_id')['family'].first(); st = W.groupby('flight_id')['subtype'].first()
    A['family'] = A['flight_id'].map(fam); A['subtype'] = A['flight_id'].map(st)
    return A


def finish_scores(S):
    Q = S[[f'q_{c}' for c in CLASSES]].to_numpy()
    S['pred'] = [CLASSES[i] for i in Q.argmax(1)]
    S['conf'] = Q.max(1)
    # coarse level: probabilities summed within each coarse class, argmax and confidence recomputed
    for cc in COARSE_CLASSES:
        S[f'qc_{cc}'] = sum(S[f'q_{c}'] for c in CLASSES if COARSE[c] == cc)
    QC = S[[f'qc_{cc}' for cc in COARSE_CLASSES]].to_numpy()
    S['coarse_family'] = S['family'].map(COARSE)
    S['coarse_pred'] = [COARSE_CLASSES[i] for i in QC.argmax(1)]
    S['coarse_conf'] = QC.max(1)
    return S


def rule_scores(A):
    """First-version fixed rule: evidence = p90 of each non-nominal class; nominal = complement; normalised."""
    S = A[['flight_id', 'family', 'subtype']].copy()
    q = np.stack([A[f'p90_{c}'].to_numpy() for c in CLASSES[1:]], 1)
    qn = np.maximum(0.0, 1.0 - q.max(1))
    v = np.column_stack([qn, q]); v = v / np.maximum(v.sum(1, keepdims=True), 1e-9)
    for i, c in enumerate(CLASSES):
        S[f'q_{c}'] = v[:, i]
    return finish_scores(S)


def oof_probs(make_model, W_tr, feats, seed, n_folds=4):
    """Out-of-fold window probabilities on the training flights (folds are flights, stratified by family)."""
    F_tr = W_tr.groupby('flight_id')['family'].first().reset_index()
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    P_oof = np.zeros((len(W_tr), len(CLASSES)))
    fid = W_tr['flight_id'].to_numpy()
    for _, ho_idx in skf.split(F_tr['flight_id'], F_tr['family']):
        ho = set(F_tr['flight_id'].iloc[ho_idx])
        m_ho = np.isin(fid, list(ho)); m_tr = ~m_ho
        model = fit(make_model(seed), W_tr.loc[m_tr, feats], W_tr.loc[m_tr, 'window_label'])
        P_oof[m_ho] = predict_proba(model, W_tr.loc[m_ho, feats])
    return P_oof


def stacked_pipeline(W_tr, feats, seed, n_folds=4):
    """Cross-fit the window model on training flights, fit the flight model on out-of-fold aggregates, then refit
    the window model on all training flights. Returns (window model, flight model, aggregate columns, temperature
    fitted on the out-of-fold training probabilities, so calibration flights are used for conformal thresholds only)."""
    P_oof = oof_probs(make_xgb, W_tr, feats, seed, n_folds)
    T = fit_temperature(P_oof, np.array([CLASSES.index(v) for v in W_tr['window_label']]))
    A_tr = flight_aggregates(W_tr, P_oof)
    agg_cols = [c for c in A_tr.columns if c not in ('flight_id', 'family', 'subtype')]
    fmodel = fit(make_flight_model(seed), A_tr[agg_cols], A_tr['family'])
    wmodel = fit(make_xgb(seed), W_tr[feats], W_tr['window_label'])
    stacked_pipeline.last_oof = P_oof
    return wmodel, fmodel, agg_cols, T


class NoveltyGate:
    """Distributional abstention: flags a flight whose aggregate evidence lies outside the calibration distribution.
    Two scorers on standardised flight aggregates fitted on training flights: Mahalanobis distance with Ledoit-Wolf
    covariance, and an isolation forest. Thresholds are the (1 - q) quantile of the calibration flights' scores, so
    in-distribution flights are withheld by the gate at rate about q."""

    def __init__(self, q=0.05, seed=0):
        self.q, self.seed = q, seed

    def fit(self, A_tr, cols):
        from sklearn.covariance import LedoitWolf
        from sklearn.ensemble import IsolationForest
        X = A_tr[cols].to_numpy(dtype=float)
        self.cols = cols
        self.med = np.nanmedian(X, 0); X = np.where(np.isnan(X), self.med, X)
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-9
        Z = (X - self.mu) / self.sd
        self.lw = LedoitWolf().fit(Z)
        self.iso = IsolationForest(n_estimators=300, random_state=self.seed).fit(Z)
        return self

    def scores(self, A):
        X = A[self.cols].to_numpy(dtype=float); X = np.where(np.isnan(X), self.med, X)
        Z = (X - self.mu) / self.sd
        return {'mahal': self.lw.mahalanobis(Z), 'iso': -self.iso.score_samples(Z)}

    def calibrate(self, A_cal):
        sc = self.scores(A_cal)
        self.th = {k: float(np.quantile(v, 1 - self.q)) for k, v in sc.items()}
        return self

    def flags(self, A):
        sc = self.scores(A)
        return {k: (sc[k] > self.th[k]) for k in sc}


def episode_starts_rule(p_non, t):
    alert = np.zeros(len(t), bool)
    for i in range(1, len(t)):
        if p_non[i] > 0.5 and p_non[i - 1] > 0.5:
            alert[i - 1] = alert[i] = True
    return [t[i] for i in range(len(t)) if alert[i] and (i == 0 or not alert[i - 1])]

def episode_starts_cusum(p_non, t, k=0.5, h=0.75):
    # one-sided CUSUM on the non-nominal probability: S_t = max(0, S_{t-1} + p_t - k); alarm while S_t > h; the
    # episode start is the first window of the excursion that led to the alarm
    S, alarm, starts, exc_start = 0.0, False, [], None
    for i in range(len(t)):
        S = max(0.0, S + p_non[i] - k)
        if S > 0 and exc_start is None:
            exc_start = i
        if S == 0:
            exc_start, alarm = None, False
        if S > h and not alarm:
            alarm = True; starts.append(t[exc_start if exc_start is not None else i])
    return starts


def stacked_scores(W, P, fmodel, agg_cols):
    A = flight_aggregates(W, P)
    S = A[['flight_id', 'family', 'subtype']].copy()
    Q = predict_proba(fmodel, A[agg_cols])
    for i, c in enumerate(CLASSES):
        S[f'q_{c}'] = Q[:, i]
    return finish_scores(S)


PROB_COLS = lambda cols: [c for c in cols if c.split('_')[0] in ('max', 'p90', 'mean', 'frac', 'run')]
RAW_COLS = lambda cols: [c for c in cols if c not in PROB_COLS(cols)]


def flight_calibration(S, level='fine'):
    """Probability calibration of the flight-level probabilities (10 equal-width bins for ECE)."""
    if level == 'fine':
        Q = S[[f'q_{c}' for c in CLASSES]].to_numpy(); y = np.array([CLASSES.index(v) for v in S['family']])
    else:
        Q = S[[f'qc_{c}' for c in COARSE_CLASSES]].to_numpy(); y = np.array([COARSE_CLASSES.index(v) for v in S['coarse_family']])
    return {'ece10': ece_top(Q, y, bins=10), 'nll': nll(Q, y), 'brier': brier(Q, y)}


# ------------------------------------------------------------------ conformal and selective prediction
def conformal_thresholds(S_cal, alpha, level='fine'):
    """Class-conditional split-conformal thresholds. level='coarse' calibrates separately on the summed coarse
    probabilities and coarse labels (a coarse set is never derived by projecting a fine set)."""
    classes, qp, fam = (CLASSES, 'q_', 'family') if level == 'fine' else (COARSE_CLASSES, 'qc_', 'coarse_family')
    th = {}
    for c in classes:
        s = 1.0 - S_cal.loc[S_cal[fam] == c, f'{qp}{c}'].to_numpy()
        n = len(s)
        if n == 0:
            th[c] = 1.0; continue
        k = int(np.ceil((n + 1) * (1 - alpha)))
        th[c] = float(np.sort(s)[min(k, n) - 1]) if k <= n else 1.0
    return th


def conformal_sets(S, th, level='fine'):
    classes, qp = (CLASSES, 'q_') if level == 'fine' else (COARSE_CLASSES, 'qc_')
    return [[c for c in classes if 1.0 - r[f'{qp}{c}'] <= th[c]] for _, r in S.iterrows()]


def conformal_report(S_test, th, tag, level='fine'):
    classes, fam = (CLASSES, 'family') if level == 'fine' else (COARSE_CLASSES, 'coarse_family')
    sets = conformal_sets(S_test, th, level)
    out = []
    for c in classes:
        m = (S_test[fam] == c).to_numpy()
        if not m.any():
            continue
        sel = [s for s, mm in zip(sets, m) if mm]
        preds = S_test.loc[m, 'pred' if level == 'fine' else 'coarse_pred'].tolist()
        out.append({'tag': tag, 'level': level, 'family': c, 'coverage': float(np.mean([c in s for s in sel])),
                    'avg_set_size': float(np.mean([len(s) for s in sel])),
                    'multi_label_rate': float(np.mean([len(s) > 1 for s in sel])),
                    'empty_rate': float(np.mean([len(s) == 0 for s in sel])),
                    'non_singleton_rate': float(np.mean([len(s) != 1 for s in sel])),
                    'verdict_outside_set_rate': float(np.mean([p not in s for p, s in zip(preds, sel)])),
                    'operational_abstain_rate': float(np.mean([(len(s) != 1) or (p not in s) for p, s in zip(preds, sel)])),
                    'n': int(m.sum())})
    return pd.DataFrame(out), sets


def risk_coverage(S):
    s = S.sort_values('conf', ascending=False)
    correct = (s['pred'] == s['family']).to_numpy().astype(float)
    cov = np.arange(1, len(s) + 1) / len(s)
    risk = 1 - np.cumsum(correct) / np.arange(1, len(s) + 1)
    trap = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz
    return pd.DataFrame({'coverage': cov, 'risk': risk}), float(trap(risk, cov))


# ------------------------------------------------------------------ helpers
def split_flights(F, rng, n_train=20, n_cal=20, n_test=20):
    """Per family: n_train training, n_cal calibration, n_test test flights; anything left over is 'extra' and is scored
    separately so the primary test set has the same per-class mixture as the calibration set."""
    tr, ca, te, ex = [], [], [], []
    for c in CLASSES:
        ids = F.loc[F.family == c, 'flight_id'].tolist(); rng.shuffle(ids)
        tr += ids[:n_train]; ca += ids[n_train:n_train + n_cal]
        te += ids[n_train + n_cal:n_train + n_cal + n_test]; ex += ids[n_train + n_cal + n_test:]
    return set(tr), set(ca), set(te), set(ex)


def window_metrics(P, y, tag):
    yi = np.array([CLASSES.index(v) for v in y])
    return {'tag': tag, 'acc': accuracy_score(yi, P.argmax(1)), 'macro_f1': f1_score(yi, P.argmax(1), average='macro'),
            'ece': ece_top(P, yi), 'nll': nll(P, yi), 'brier': brier(P, yi)}


def sub(W, ids):
    return W[W.flight_id.isin(ids)].reset_index(drop=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--features', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--reps', type=int, default=5)
    ap.add_argument('--only', default='all', help='comma-separated subset of e0,e1,e2,e3 to run (others are skipped; existing files kept)')
    ap.add_argument('--e2_reps', type=int, default=3, help='seeds for the unseen-subtype experiment')
    ap.add_argument('--real_features', default=None, help='features folder for real PX4 logs without ground truth (E6)')
    ap.add_argument('--e7_epochs', type=int, default=6, help='training epochs for the sequence baseline (E7)')
    args = ap.parse_args()
    RUN = set(['e0', 'e1', 'e2', 'e3', 'e5', 'e7']) if args.only == 'all' else set(args.only.split(','))
    if args.real_features and args.only == 'all':
        RUN.add('e6')
    fdir, out = pathlib.Path(args.features), pathlib.Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    W = engineer(pd.read_csv(fdir / 'windows.csv'))
    feats = [c for c in W.columns if c not in META and not c.startswith('rx_') and pd.api.types.is_numeric_dtype(W[c])]
    Ws = W[W.source == 'sih'].reset_index(drop=True)
    Ww = W[W.source == 'whelan'].reset_index(drop=True)
    F = Ws.groupby('flight_id').agg(family=('family', 'first'), subtype=('subtype', 'first')).reset_index()
    print(f'features used: {len(feats)} (base + context + flight-relative)')
    print('SIH windows:', len(Ws), '| flights:', len(F), '| Whelan windows:', len(Ww))
    print('window labels:', Ws.window_label.value_counts().to_dict())

    # ---------------- E0 leakage audit, paired with matched training-window counts
    e0 = []
    for rep in (range(args.reps) if 'e0' in RUN else []):
        rng = np.random.RandomState(500 + rep)
        tr, ca, te, ex = split_flights(F, rng, n_train=30, n_cal=0, n_test=10 ** 6)
        W_tr, W_te = sub(Ws, tr), sub(Ws, te)
        m = fit(make_xgb(rep), W_tr[feats], W_tr['window_label'])
        g = window_metrics(predict_proba(m, W_te[feats]), W_te['window_label'], 'flight_grouped_split'); g['rep'] = rep; g['n_train_windows'] = len(W_tr)
        idx = rng.permutation(len(Ws)); k = len(W_tr)
        m = fit(make_xgb(rep), Ws.loc[idx[:k], feats], Ws.loc[idx[:k], 'window_label'])
        r = window_metrics(predict_proba(m, Ws.loc[idx[k:], feats]), Ws.loc[idx[k:], 'window_label'], 'random_window_split'); r['rep'] = rep; r['n_train_windows'] = k
        e0 += [g, r]
    if e0:
        E0 = pd.DataFrame(e0); E0.to_csv(out / 'e0_leakage_audit.csv', index=False)
        piv = E0.pivot(index='rep', columns='tag', values='acc'); d = piv['random_window_split'] - piv['flight_grouped_split']
        print('\n[E0] leakage audit (window-level), mean over paired reps:\n', E0.groupby('tag')[['acc', 'macro_f1', 'ece', 'nll', 'brier']].mean().round(4).to_string())
        print(f'[E0] paired accuracy difference random - grouped: {d.mean():.4f} ± {d.std(ddof=0):.4f} over {len(d)} reps')

    # ---------------- E1 in-distribution, flight-grouped, repeated
    GAP_FEATS = [c for c in feats if c.startswith('baro_gap_frac') or c.startswith('mag_gap_frac')]
    feats_nogap = [c for c in feats if c not in GAP_FEATS]
    wm, fm, cf, rc_all, extra_rows = [], [], [], [], []
    for rep in (range(args.reps) if 'e1' in RUN else []):
        rng = np.random.RandomState(100 + rep)
        tr, ca, te, ex = split_flights(F, rng)
        W_tr, W_ca, W_te, W_ex = sub(Ws, tr), sub(Ws, ca), sub(Ws, te), sub(Ws, ex)
        # full stacked pipeline (its out-of-fold window probabilities also give the temperature for the window and rule branches)
        wmodel, fmodel, agg_cols, T = stacked_pipeline(W_tr, feats, rep)
        for name, mk in (('xgb', make_xgb), ('logreg', make_logreg)):
            model = wmodel if name == 'xgb' else fit(mk(rep), W_tr[feats], W_tr['window_label'])
            P_te = predict_proba(model, W_te[feats])
            T_m = T if name == 'xgb' else fit_temperature(oof_probs(mk, W_tr, feats, rep), np.array([CLASSES.index(v) for v in W_tr['window_label']]))
            wm.append({'rep': rep, 'model': name, 'T': T_m, **window_metrics(P_te, W_te['window_label'], 'raw')})
            wm.append({'rep': rep, 'model': name, 'T': T_m, **window_metrics(apply_temperature(P_te, T_m), W_te['window_label'], 'temp_scaled')})
        P_oof = stacked_pipeline.last_oof
        A_tr = flight_aggregates(W_tr, P_oof)
        f_probs = fit(make_flight_model(rep), A_tr[PROB_COLS(agg_cols)], A_tr['family'])
        f_raw = fit(make_flight_model(rep), A_tr[RAW_COLS(agg_cols)], A_tr['family'])
        P_cal, P_te, P_ex = predict_proba(wmodel, W_ca[feats]), predict_proba(wmodel, W_te[feats]), (predict_proba(wmodel, W_ex[feats]) if len(W_ex) else None)
        P_cal_t, P_te_t = apply_temperature(P_cal, T), apply_temperature(P_te, T)
        # sensitivity: full stacked pipeline without the derived-topic gap features
        wmodel_ng, fmodel_ng, agg_cols_ng, _ = stacked_pipeline(W_tr, feats_nogap, rep)
        variants = {
            'rule': (rule_scores(flight_aggregates(W_ca, P_cal_t)), rule_scores(flight_aggregates(W_te, P_te_t)),
                     rule_scores(flight_aggregates(W_ex, apply_temperature(P_ex, T))) if len(W_ex) else None),
            'stacked_probs_only': (None, None, None), 'stacked_raw_only': (None, None, None),
            'stacked': (stacked_scores(W_ca, P_cal, fmodel, agg_cols), stacked_scores(W_te, P_te, fmodel, agg_cols),
                        stacked_scores(W_ex, P_ex, fmodel, agg_cols) if len(W_ex) else None),
            'stacked_no_gap': (stacked_scores(W_ca, predict_proba(wmodel_ng, W_ca[feats_nogap]), fmodel_ng, agg_cols_ng),
                               stacked_scores(W_te, predict_proba(wmodel_ng, W_te[feats_nogap]), fmodel_ng, agg_cols_ng), None),
        }
        for tag, fm_, cols in (('stacked_probs_only', f_probs, PROB_COLS(agg_cols)), ('stacked_raw_only', f_raw, RAW_COLS(agg_cols))):
            def sc(W, P):
                A = flight_aggregates(W, P); S = A[['flight_id', 'family', 'subtype']].copy(); Q = predict_proba(fm_, A[cols])
                for i, c in enumerate(CLASSES):
                    S[f'q_{c}'] = Q[:, i]
                return finish_scores(S)
            variants[tag] = (sc(W_ca, P_cal), sc(W_te, P_te), None)
        thresholds = {}
        for agg, (S_cal, S_te, S_ex) in variants.items():
            rcurve, aurc = risk_coverage(S_te)
            cal_f, cal_c = flight_calibration(S_te, 'fine'), flight_calibration(S_te, 'coarse')
            fm.append({'rep': rep, 'aggregation': agg, 'flight_acc': accuracy_score(S_te.family, S_te.pred),
                       'flight_macro_f1': f1_score(S_te.family, S_te.pred, average='macro'), 'aurc': aurc,
                       'coarse_acc': accuracy_score(S_te.coarse_family, S_te.coarse_pred),
                       'coarse_macro_f1': f1_score(S_te.coarse_family, S_te.coarse_pred, average='macro'),
                       'flight_ece10': cal_f['ece10'], 'flight_nll': cal_f['nll'], 'flight_brier': cal_f['brier'],
                       'coarse_ece10': cal_c['ece10'], 'coarse_nll': cal_c['nll'], 'coarse_brier': cal_c['brier']})
            rcurve['rep'] = rep; rcurve['aggregation'] = agg; rc_all.append(rcurve)
            if agg in ('rule', 'stacked'):
                for alpha in (0.10, 0.20):
                    for level in ('fine', 'coarse'):
                        th = conformal_thresholds(S_cal, alpha, level)
                        thresholds[f'{agg}_{level}_alpha{alpha}'] = th
                        rep_df, sets = conformal_report(S_te, th, f'{agg}_alpha{alpha}', level)
                        rep_df['rep'] = rep; rep_df['aggregation'] = agg; rep_df['alpha'] = alpha
                        cf.append(rep_df)
                        S_te[f'set_{level}_a{alpha}'] = [','.join(x) for x in sets]
                        if S_ex is not None and len(S_ex):
                            ex_df, _ = conformal_report(S_ex, th, f'{agg}_alpha{alpha}', level)
                            ex_df['rep'] = rep; ex_df['aggregation'] = agg; ex_df['alpha'] = alpha
                            ex_df['acc'] = accuracy_score(S_ex.family, S_ex.pred) if level == 'fine' else accuracy_score(S_ex.coarse_family, S_ex.coarse_pred)
                            extra_rows.append(ex_df)
                S_te.assign(rep=rep, aggregation=agg).to_csv(out / f'e1_rep{rep}_{agg}_flight_scores.csv', index=False)
                S_cal.assign(rep=rep, aggregation=agg).to_csv(out / f'e1_rep{rep}_{agg}_calibration_scores.csv', index=False)
        (out / f'e1_rep{rep}_thresholds.json').write_text(json.dumps(thresholds, indent=1))
        if rep == 0:
            imp = getattr(wmodel, 'feature_importances_', None)
            if imp is not None:
                pd.DataFrame({'feature': feats, 'importance': imp}).sort_values('importance', ascending=False)\
                    .to_csv(out / 'e1_rep0_window_feature_importance.csv', index=False)
            lr = fmodel[-1]
            pd.DataFrame(lr.coef_, index=[CLASSES[int(i)] for i in lr.classes_], columns=agg_cols).T\
                .to_csv(out / 'e1_rep0_flight_model_coefficients.csv')
    if not wm:
        WM = FM = CF = None
    else:
        WM, FM, CF = pd.DataFrame(wm), pd.DataFrame(fm), pd.concat(cf, ignore_index=True)
    if WM is not None:
        WM.to_csv(out / 'e1_window_metrics.csv', index=False); FM.to_csv(out / 'e1_flight_metrics.csv', index=False)
    if WM is not None:
        CF.to_csv(out / 'e1_conformal.csv', index=False); pd.concat(rc_all).to_csv(out / 'e1_risk_coverage.csv', index=False)
    if extra_rows:
        pd.concat(extra_rows, ignore_index=True).to_csv(out / 'e1_extra_flights.csv', index=False)
    if WM is not None:
        print('\n[E1] window metrics, mean over reps:\n', WM.groupby(['model', 'tag'])[['acc', 'macro_f1', 'ece', 'nll', 'brier']].mean().round(4).to_string())
        print('\n[E1] flight-level, mean (std) over reps:\n', FM.groupby('aggregation')[['flight_acc', 'flight_macro_f1', 'aurc', 'coarse_acc', 'coarse_macro_f1']].agg(['mean', 'std']).round(3).to_string())
        print('\n[E1] flight-level probability calibration, mean over reps:\n', FM.groupby('aggregation')[['flight_ece10', 'flight_nll', 'flight_brier', 'coarse_ece10', 'coarse_nll', 'coarse_brier']].mean().round(3).to_string())
        print('\n[E1] class-conditional conformal at flight level, mean (std) over reps:\n',
              CF.groupby(['aggregation', 'level', 'alpha', 'family'])[['coverage', 'avg_set_size', 'multi_label_rate', 'empty_rate', 'non_singleton_rate', 'verdict_outside_set_rate', 'operational_abstain_rate']].agg(['mean', 'std']).round(3).to_string())
    if extra_rows:
        EX = pd.concat(extra_rows, ignore_index=True)
        print('\n[E1] extra flights beyond the balanced test set (scored with the same thresholds), mean over reps:\n',
              EX.groupby(['aggregation', 'level', 'alpha', 'family'])[['n', 'acc', 'coverage', 'operational_abstain_rate']].mean().round(3).to_string())

    # ---------------- E2 leave-one-subtype-out: ONE fitted pipeline per hold-out and seed, evaluated on a disjoint
    # seen-subtype test set and on the withheld subtype, so model, training volume and thresholds are held fixed
    def score_set(S, fam, S_cal):
        cfam = COARSE[fam]
        row = {'n': len(S), 'acc': float((S.pred == fam).mean()), 'coarse_acc': float((S.coarse_pred == cfam).mean()),
               'mean_conf': float(S.conf.mean()), 'pred_dist': json.dumps(S.pred.value_counts().to_dict())}
        for alpha in (0.10, 0.20):
            th = conformal_thresholds(S_cal, alpha); sets = conformal_sets(S, th)
            row[f'coverage_a{alpha}'] = float(np.mean([fam in x for x in sets]))
            row[f'operational_abstain_a{alpha}'] = float(np.mean([(len(x) != 1) or (p not in x) for p, x in zip(S.pred, sets)]))
            thc = conformal_thresholds(S_cal, alpha, 'coarse'); csets = conformal_sets(S, thc, 'coarse')
            row[f'coarse_coverage_a{alpha}'] = float(np.mean([cfam in x for x in csets]))
            row[f'coarse_operational_abstain_a{alpha}'] = float(np.mean([(len(x) != 1) or (p not in x) for p, x in zip(S.coarse_pred, csets)]))
        return row

    per_seed, splits, gate_scores_all = [], [], []
    E2_SEEDS = list(range(args.e2_reps))
    for fam in (CLASSES[1:] if 'e2' in RUN else []):
        for st in sorted(F.loc[F.family == fam, 'subtype'].unique()):
            held = F.loc[F.subtype == st, 'flight_id'].tolist()
            for seed in E2_SEEDS:
                rng = np.random.RandomState(700 + seed)
                seen = F.loc[(F.family == fam) & (F.subtype != st), 'flight_id'].tolist(); rng.shuffle(seen)
                n_tr = len(seen) // 3; n_ca = len(seen) // 3
                tr, ca = seen[:n_tr], seen[n_tr:n_tr + n_ca]; seen_test = seen[n_tr + n_ca:]
                alloc = {fam: (n_tr, n_ca, len(seen_test))}
                for c in CLASSES:
                    if c == fam:
                        continue
                    ids = F.loc[F.family == c, 'flight_id'].tolist(); rng.shuffle(ids)
                    tr += ids[:20]; ca += ids[20:40]; alloc[c] = (20, 20, 0)
                W_tr, W_ca = sub(Ws, tr), sub(Ws, ca)
                wmodel, fmodel, agg_cols, T = stacked_pipeline(W_tr, feats, 700 + seed)
                A_tr = flight_aggregates(W_tr, stacked_pipeline.last_oof)
                A_ca = flight_aggregates(W_ca, predict_proba(wmodel, W_ca[feats]))
                A_seen = flight_aggregates(sub(Ws, seen_test), predict_proba(wmodel, sub(Ws, seen_test)[feats]))
                A_held = flight_aggregates(sub(Ws, held), predict_proba(wmodel, sub(Ws, held)[feats]))
                gate = NoveltyGate(q=0.05, seed=700 + seed).fit(A_tr, agg_cols).calibrate(A_ca)
                # declared rule baseline for the gate: withhold when any availability or gap aggregate exceeds its
                # 95th percentile on the calibration flights; the rule score is the largest ratio to that percentile
                gap_cols = [c for c in agg_cols if c.endswith('_frac_max')]
                gap_q95 = {c: float(np.nanquantile(A_ca[c].to_numpy(dtype=float), 0.95)) for c in gap_cols}
                def gap_rule_score(A_):
                    X = A_[gap_cols].to_numpy(dtype=float); X = np.where(np.isnan(X), 0.0, X)
                    q = np.array([gap_q95[c] for c in gap_cols]) + 1e-6
                    return (X / q).max(axis=1)
                S_cal = stacked_scores(W_ca, predict_proba(wmodel, W_ca[feats]), fmodel, agg_cols)
                S_seen = stacked_scores(sub(Ws, seen_test), predict_proba(wmodel, sub(Ws, seen_test)[feats]), fmodel, agg_cols)
                S_held = stacked_scores(sub(Ws, held), predict_proba(wmodel, sub(Ws, held)[feats]), fmodel, agg_cols)
                r_seen, r_held = score_set(S_seen, fam, S_cal), score_set(S_held, fam, S_cal)
                # distributional abstention gate: fraction withheld by each scorer, and combined with the conformal policy
                for tag_, S_, A_, r_ in (('seen', S_seen, A_seen, r_seen), ('unseen', S_held, A_held, r_held)):
                    fl = gate.flags(A_)
                    th = conformal_thresholds(S_cal, 0.10); sets = conformal_sets(S_, th)
                    conf_abst = np.array([(len(x) != 1) or (p not in x) for p, x in zip(S_.pred, sets)])
                    for k in ('mahal', 'iso'):
                        r_[f'gate_{k}_withheld'] = float(fl[k].mean())
                        r_[f'gate_{k}_or_conformal_withheld'] = float((fl[k] | conf_abst).mean())
                    S_['gate_mahal'] = fl['mahal']; S_['gate_iso'] = fl['iso']
                    rule_fl = gap_rule_score(A_) > 1.0
                    r_['gate_gaprule_withheld'] = float(rule_fl.mean())
                    r_['gate_gaprule_or_conformal_withheld'] = float((rule_fl | conf_abst).mean())
                    S_['gate_gaprule'] = rule_fl
                # threshold-free separability of each novelty score (and of plain confidence) between the withheld subtype and
                # the seen-subtype test set: AUROC per seed, and the catch rate at a fixed 5% false-abstention budget on seen flights
                from sklearn.metrics import roc_auc_score
                sc_seen, sc_un = gate.scores(A_seen), gate.scores(A_held)
                sc_seen['one_minus_conf'] = 1.0 - S_seen['conf'].to_numpy(); sc_un['one_minus_conf'] = 1.0 - S_held['conf'].to_numpy()
                sc_seen['gaprule'] = gap_rule_score(A_seen); sc_un['gaprule'] = gap_rule_score(A_held)
                for k in ('mahal', 'iso', 'one_minus_conf', 'gaprule'):
                    y = np.r_[np.ones(len(sc_un[k])), np.zeros(len(sc_seen[k]))]; x = np.r_[sc_un[k], sc_seen[k]]
                    r_held[f'gate_{k}_auroc'] = float(roc_auc_score(y, x)) if len(sc_un[k]) and len(sc_seen[k]) else np.nan
                    cut = float(np.quantile(sc_seen[k], 0.95)) if len(sc_seen[k]) else np.nan
                    r_held[f'gate_{k}_catch_at_5pct_fpr'] = float(np.mean(sc_un[k] > cut)) if len(sc_un[k]) else np.nan
                gate_rows = []
                for set_name, S_, sc in (('seen_test', S_seen, sc_seen), ('unseen', S_held, sc_un),
                                         ('calibration', S_cal, {**gate.scores(A_ca), 'one_minus_conf': 1.0 - S_cal['conf'].to_numpy(), 'gaprule': gap_rule_score(A_ca)})):
                    for i, fid in enumerate(S_['flight_id']):
                        gate_rows.append({'held_out_subtype': st, 'seed': seed, 'set': set_name, 'flight_id': fid, 'family': S_['family'].iloc[i],
                                          'mahal': float(sc['mahal'][i]), 'iso': float(sc['iso'][i]), 'one_minus_conf': float(sc['one_minus_conf'][i]),
                                          'gaprule': float(sc['gaprule'][i])})
                gate_scores_all.extend(gate_rows)
                per_seed.append({'held_out_family': fam, 'held_out_subtype': st, 'seed': seed,
                                 **{f'unseen_{k}': v for k, v in r_held.items()}, **{f'seen_{k}': v for k, v in r_seen.items()}})
                for c in CLASSES:
                    n_ca_c = alloc[c][1]
                    splits.append({'held_out_subtype': st, 'seed': seed, 'family': c, 'n_train': alloc[c][0], 'n_cal': n_ca_c,
                                   'n_seen_test': alloc[c][2], 'rank_alpha0.1': int(np.ceil((n_ca_c + 1) * 0.9)),
                                   'rank_alpha0.2': int(np.ceil((n_ca_c + 1) * 0.8)), 'rank_at_max_alpha0.1': int(np.ceil((n_ca_c + 1) * 0.9)) >= n_ca_c})
                if seed == 0:
                    S_held.assign(held_out_subtype=st).to_csv(out / f'e2_held_{st}_flight_scores.csv', index=False)
                    pd.DataFrame({'flight_id': A_ca['flight_id'], **gate.scores(A_ca)}).to_csv(out / f'e2_gate_cal_scores_{st}.csv', index=False)
                    S_seen.assign(held_out_subtype=st).to_csv(out / f'e2_seen_{st}_flight_scores.csv', index=False)
    if per_seed:
        PS = pd.DataFrame(per_seed); PS.to_csv(out / 'e2_per_seed.csv', index=False)
        num = [c for c in PS.columns if c not in ('held_out_family', 'held_out_subtype', 'seed', 'unseen_pred_dist', 'seen_pred_dist')]
        g = PS.groupby(['held_out_family', 'held_out_subtype'], sort=False)
        LOSO = g[num].mean().round(4); LOSO_sd = g[num].std(ddof=0).round(4)
        LOSO.columns = [f'{c}_mean' for c in LOSO.columns]; LOSO_sd.columns = [f'{c}_std' for c in LOSO_sd.columns]
        LOSO = pd.concat([LOSO, LOSO_sd], axis=1).reset_index()
        LOSO['unseen_pred_dist_seed0'] = g['unseen_pred_dist'].first().to_numpy()
        LOSO.to_csv(out / 'e2_leave_one_subtype_out.csv', index=False)
        pd.DataFrame(splits).to_csv(out / 'e2_splits.csv', index=False)
        pd.DataFrame(gate_scores_all).to_csv(out / 'e2_gate_scores.csv', index=False)
        show = ['held_out_family', 'held_out_subtype', 'unseen_n_mean', 'seen_n_mean', 'unseen_acc_mean', 'seen_acc_mean', 'unseen_coarse_acc_mean',
                'seen_coarse_acc_mean', 'unseen_mean_conf_mean', 'seen_mean_conf_mean', 'unseen_coverage_a0.1_mean', 'seen_coverage_a0.1_mean',
                'unseen_coarse_coverage_a0.1_mean', 'seen_coarse_coverage_a0.1_mean', 'unseen_operational_abstain_a0.1_mean']
        gate_show = ['held_out_subtype', 'unseen_gate_mahal_withheld_mean', 'seen_gate_mahal_withheld_mean', 'unseen_gate_iso_withheld_mean', 'seen_gate_iso_withheld_mean',
                     'unseen_gate_gaprule_withheld_mean', 'seen_gate_gaprule_withheld_mean',
                     'unseen_gate_mahal_or_conformal_withheld_mean', 'seen_gate_mahal_or_conformal_withheld_mean', 'unseen_operational_abstain_a0.1_mean', 'seen_operational_abstain_a0.1_mean']
        sep_show = ['held_out_subtype', 'unseen_gate_mahal_auroc_mean', 'unseen_gate_mahal_auroc_std', 'unseen_gate_mahal_catch_at_5pct_fpr_mean', 'unseen_gate_iso_auroc_mean',
                    'unseen_gate_iso_catch_at_5pct_fpr_mean', 'unseen_gate_one_minus_conf_auroc_mean', 'unseen_gate_one_minus_conf_catch_at_5pct_fpr_mean',
                    'unseen_gate_gaprule_auroc_mean', 'unseen_gate_gaprule_catch_at_5pct_fpr_mean']
        print(f'\n[E2] leave-one-subtype-out, one fitted pipeline per hold-out evaluated on the withheld subtype and on a disjoint seen-subtype test set, mean over {len(E2_SEEDS)} seeds:\n',
              LOSO[show].round(3).to_string(index=False))
        print(f'\n[E4] distributional abstention gate (threshold at the 95th percentile of calibration scores), fraction withheld, mean over {len(E2_SEEDS)} seeds:\n',
              LOSO[gate_show].round(3).to_string(index=False))
        print(f'\n[E4] threshold-free separability, withheld subtype against seen-subtype test set (AUROC per seed then averaged; catch rate at the score cut that withholds 5% of seen flights):\n',
              LOSO[sep_show].round(3).to_string(index=False))

    # ---------------- E5 onset localisation against the logged onset (one fitted pipeline, held-out test flights)
    if 'e5' in RUN:
        tr, ca, te, ex = split_flights(F, np.random.RandomState(900), n_train=20, n_cal=20, n_test=10 ** 6)
        W_tr, W_te = sub(Ws, tr), sub(Ws, te | ex)
        wmodel, fmodel, agg_cols, T = stacked_pipeline(W_tr, feats, 900)
        P_te = apply_temperature(predict_proba(wmodel, W_te[feats]), T)
        rows = []
        for fid, idx in W_te.groupby('flight_id', sort=False).indices.items():
            d = W_te.iloc[idx].sort_values('t_start'); p_non = 1.0 - P_te[idx][np.argsort(W_te.iloc[idx]['t_start'].to_numpy()), 0]
            t = d['t_start'].to_numpy(); fam = d['family'].iloc[0]; st = d['subtype'].iloc[0]
            onset = float(d['onset_s'].iloc[0]) if fam != 'nominal' and pd.notna(d['onset_s'].iloc[0]) else np.nan
            restore = float(d['restore_s'].iloc[0]) if pd.notna(d['restore_s'].iloc[0]) else np.nan
            for method, starts in (('rule', episode_starts_rule(p_non, t)), ('cusum', episode_starts_cusum(p_non, t))):
                row = {'flight_id': fid, 'method': method, 'family': fam, 'subtype': st, 'onset_s': onset, 'restore_s': restore,
                       'n_alert_episodes': len(starts), 'first_alert_s': starts[0] if starts else np.nan}
                if fam == 'nominal':
                    row.update({'false_alert': bool(starts), 'est_onset_s': np.nan, 'onset_error_s': np.nan, 'early_alert': bool(starts)})
                else:
                    after = [x for x in starts if x >= onset - 5.0]          # first episode starting at or after onset (5 s tolerance)
                    est = after[0] if after else np.nan
                    row.update({'est_onset_s': est, 'onset_error_s': (est - onset) if after else np.nan,
                                'detected_within_60s': bool(after) and (est - onset) <= 60.0,
                                'early_alert': any(x < onset - 5.0 for x in starts), 'false_alert': np.nan})
                rows.append(row)
        E5 = pd.DataFrame(rows); E5.to_csv(out / 'e5_onset_localisation.csv', index=False)
        att = E5[E5.family != 'nominal']
        summ = att.groupby(['method', 'family', 'subtype']).apply(lambda g: pd.Series({
            'n': len(g), 'detected_within_60s': float(g['detected_within_60s'].fillna(False).mean()),
            'median_abs_error_s': float(g['onset_error_s'].abs().median()) if g['onset_error_s'].notna().any() else np.nan,
            'within_10s': float((g['onset_error_s'].abs() <= 10.0).mean()), 'within_30s': float((g['onset_error_s'].abs() <= 30.0).mean()), 'median_signed_error_s': float(g['onset_error_s'].median()) if g['onset_error_s'].notna().any() else np.nan,
            'early_alert_rate': float(g['early_alert'].mean())})).reset_index()
        nom = E5[E5.family == 'nominal']
        summ = pd.concat([summ] + [pd.DataFrame([{'method': m, 'family': 'nominal', 'subtype': 'none', 'n': int((nom.method == m).sum()), 'detected_within_60s': np.nan,
                                                   'median_abs_error_s': np.nan, 'within_10s': np.nan, 'within_30s': np.nan, 'median_signed_error_s': np.nan,
                                                   'early_alert_rate': float(nom.loc[nom.method == m, 'false_alert'].mean())}]) for m in ('rule', 'cusum')], ignore_index=True)
        summ.to_csv(out / 'e5_onset_summary.csv', index=False)
        print('\n[E5] onset localisation against the logged onset (one fitted pipeline): declared rule and CUSUM baseline (k = 0.5, h = 0.75); early alert = any episode starting more than 5 s before onset; for nominal flights the last column is the false-alert rate:\n',
              summ.round(3).to_string(index=False))

    # ---------------- E6 real PX4 logs (no ground truth): frozen pipeline, verdict distribution, abstention, gate
    if 'e6' in RUN and args.real_features:
        Wr_raw = pd.read_csv(pathlib.Path(args.real_features) / 'windows.csv')
        Wr_raw = Wr_raw[Wr_raw.source == 'real'].reset_index(drop=True)
        # drop duplicate uploads: flights whose window feature matrices are identical (same log under several ids)
        sig_cols = [c for c in ('t_start', 'posvel_rms', 'gps_gap_frac', 'gps_baro_dz_rms', 'baro_raw_std', 'mag_raw_std', 'step_speed_max') if c in Wr_raw.columns]
        import hashlib
        def flight_signature(d):
            arr = np.nan_to_num(np.round(d.to_numpy(dtype=float), 6), nan=-999.0)
            return hashlib.sha1(arr.tobytes()).hexdigest()
        sig = Wr_raw.groupby('flight_id')[sig_cols].apply(flight_signature).rename('signature').reset_index()
        keep_ids = sig.drop_duplicates(subset=['signature'])['flight_id']
        n_dup = int(Wr_raw.flight_id.nunique() - len(keep_ids))
        Wr_raw = Wr_raw[Wr_raw.flight_id.isin(set(keep_ids))].reset_index(drop=True)
        print(f'[E6] duplicate uploads removed: {n_dup} (identical window feature matrices); flights kept: {Wr_raw.flight_id.nunique()}')
        Wr = engineer(Wr_raw)
        for c in feats:
            if c not in Wr.columns:
                Wr[c] = np.nan
        Fr = pd.read_csv(pathlib.Path(args.real_features) / 'flights.csv') if (pathlib.Path(args.real_features) / 'flights.csv').exists() else None
        tr, ca, te, ex = split_flights(F, np.random.RandomState(0), n_train=40, n_cal=20)
        W_tr, W_ca = sub(Ws, tr), sub(Ws, ca)
        wmodel, fmodel, agg_cols, T = stacked_pipeline(W_tr, feats, 0)
        A_tr = flight_aggregates(W_tr, stacked_pipeline.last_oof)
        P_ca = predict_proba(wmodel, W_ca[feats]); A_ca = flight_aggregates(W_ca, P_ca)
        gate = NoveltyGate(q=0.05, seed=0).fit(A_tr, agg_cols).calibrate(A_ca)
        gap_cols = [c for c in agg_cols if c.endswith('_frac_max')]
        gap_q95 = {c: float(np.nanquantile(A_ca[c].to_numpy(dtype=float), 0.95)) for c in gap_cols}
        S_cal = stacked_scores(W_ca, P_ca, fmodel, agg_cols)
        th10, thc10 = conformal_thresholds(S_cal, 0.10), conformal_thresholds(S_cal, 0.10, 'coarse')
        base_cols = [c for c in feats if not (c.endswith('_d1') or c.endswith('_rmax3') or c.endswith('_rel'))]
        miss = Wr.groupby('flight_id')[base_cols].apply(lambda d: float(d.isna().to_numpy().mean())).rename('missing_frac')
        Pr = predict_proba(wmodel, Wr[feats]); Pr_t = apply_temperature(Pr, T)
        A_r = flight_aggregates(Wr, Pr)
        S_r = stacked_scores(Wr, Pr, fmodel, agg_cols)
        sets_f, sets_c = conformal_sets(S_r, th10), conformal_sets(S_r, thc10, 'coarse')
        S_r['set_a0.10'] = [','.join(x) for x in sets_f]; S_r['coarse_set_a0.10'] = [','.join(x) for x in sets_c]
        S_r['abstain_fine'] = [(len(x) != 1) or (p not in x) for p, x in zip(S_r.pred, sets_f)]
        S_r['abstain_coarse'] = [(len(x) != 1) or (p not in x) for p, x in zip(S_r.coarse_pred, sets_c)]
        fl = gate.flags(A_r); S_r['gate_mahal'] = fl['mahal']; S_r['gate_iso'] = fl['iso']
        sc = gate.scores(A_r); S_r['mahal_score'] = sc['mahal']; S_r['iso_score'] = sc['iso']
        Xg = A_r[gap_cols].to_numpy(dtype=float); Xg = np.where(np.isnan(Xg), 0.0, Xg)
        S_r['gate_gaprule'] = (Xg / (np.array([gap_q95[c] for c in gap_cols]) + 1e-6)).max(axis=1) > 1.0
        # window-level alert episodes (declared rule) and predicted-class fractions per flight
        ep, fr = {}, {}
        for fid, idx in Wr.groupby('flight_id', sort=False).indices.items():
            order = np.argsort(Wr.iloc[idx]['t_start'].to_numpy()); p_non = 1.0 - Pr_t[idx][order, 0]
            t = Wr.iloc[idx]['t_start'].to_numpy()[order]
            alert = np.zeros(len(t), bool)
            for i in range(1, len(t)):
                if p_non[i] > 0.5 and p_non[i - 1] > 0.5:
                    alert[i - 1] = alert[i] = True
            ep[fid] = int(sum(1 for i in range(len(t)) if alert[i] and (i == 0 or not alert[i - 1])))
            fr[fid] = {c: float((Pr_t[idx].argmax(1) == j).mean()) for j, c in enumerate(CLASSES)}
        S_r['alert_episodes'] = S_r['flight_id'].map(ep)
        S_r['missing_frac'] = S_r['flight_id'].map(miss)
        for c in CLASSES:
            S_r[f'window_frac_{c}'] = S_r['flight_id'].map(lambda f: fr[f][c])
        if Fr is not None and 'flight_id' in Fr.columns:
            keep = [c for c in ('duration_s', 'n_windows', 'ver_sw', 'ver_sw_release', 'ver_hw') if c in Fr.columns]
            S_r = S_r.merge(Fr[['flight_id'] + keep], on='flight_id', how='left')
            if 'ver_sw_release' in S_r.columns:
                def decode_release(v):
                    try:
                        v = int(float(v))
                    except (TypeError, ValueError):
                        return 'unknown'
                    major, minor = (v >> 24) & 0xFF, (v >> 16) & 0xFF
                    return f'v{major}.{minor}' if major > 0 else 'unknown'
                S_r['px4_version'] = S_r['ver_sw_release'].map(decode_release)
        S_r.to_csv(out / 'e6_real_flight_verdicts.csv', index=False)
        summ = {'n_flights': int(len(S_r)), 'n_windows': int(len(Wr)), 'duplicates_removed': n_dup,
                **{f'verdict_frac_{c}': float((S_r.pred == c).mean()) for c in CLASSES},
                **{f'coarse_verdict_frac_{c}': float((S_r.coarse_pred == c).mean()) for c in COARSE_CLASSES},
                'abstain_fine_rate': float(S_r.abstain_fine.mean()), 'abstain_coarse_rate': float(S_r.abstain_coarse.mean()),
                'gate_mahal_withheld': float(S_r.gate_mahal.mean()), 'gate_iso_withheld': float(S_r.gate_iso.mean()),
                'gate_gaprule_withheld': float(S_r.gate_gaprule.mean()),
                'mean_conf': float(S_r.conf.mean()), 'median_conf': float(S_r.conf.median()),
                'frac_with_alert_episode': float((S_r.alert_episodes > 0).mean()), 'mean_alert_episodes': float(S_r.alert_episodes.mean()),
                'nominal_and_not_withheld_frac': float(((S_r.pred == 'nominal') & ~S_r.abstain_fine & ~S_r.gate_mahal).mean()),
                'missing_frac_mean': float(S_r.missing_frac.mean()), 'missing_frac_median': float(S_r.missing_frac.median()),
                'missing_frac_max': float(S_r.missing_frac.max()),
                'gate_withheld_low_missing': float(S_r.loc[S_r.missing_frac <= 0.05, 'gate_mahal'].mean()) if (S_r.missing_frac <= 0.05).any() else np.nan,
                'gate_withheld_high_missing': float(S_r.loc[S_r.missing_frac > 0.05, 'gate_mahal'].mean()) if (S_r.missing_frac > 0.05).any() else np.nan,
                'n_low_missing': int((S_r.missing_frac <= 0.05).sum()), 'n_high_missing': int((S_r.missing_frac > 0.05).sum()),
                'abstain_low_missing': float(S_r.loc[S_r.missing_frac <= 0.05, 'abstain_fine'].mean()) if (S_r.missing_frac <= 0.05).any() else np.nan,
                'abstain_high_missing': float(S_r.loc[S_r.missing_frac > 0.05, 'abstain_fine'].mean()) if (S_r.missing_frac > 0.05).any() else np.nan,
                'nominal_low_missing': float((S_r.loc[S_r.missing_frac <= 0.05, 'pred'] == 'nominal').mean()) if (S_r.missing_frac <= 0.05).any() else np.nan,
                'nominal_high_missing': float((S_r.loc[S_r.missing_frac > 0.05, 'pred'] == 'nominal').mean()) if (S_r.missing_frac > 0.05).any() else np.nan}
        pd.DataFrame([summ]).to_csv(out / 'e6_real_summary.csv', index=False)
        print('\n[E6] real PX4 logs (no ground truth), frozen pipeline trained on simulation only:')
        for k, v in summ.items():
            print(f'  {k:32s} {v:.3f}' if isinstance(v, float) else f'  {k:32s} {v}')
        if 'px4_version' in S_r.columns:
            by = S_r.groupby('px4_version').agg(n=('flight_id', 'size'), nominal=('pred', lambda x: float((x == 'nominal').mean())),
                                                abstain=('abstain_fine', 'mean'), gate=('gate_mahal', 'mean'), conf=('conf', 'mean')).round(3)
            by.index.name = 'px4_version'
            by.to_csv(out / 'e6_real_by_version.csv'); print('\n[E6] by PX4 release (major.minor):\n', by.to_string())

    # ---------------- E7 published architectures under the flight-level protocol
    if 'e7' in RUN:
        try:
            import torch, torch.nn as nn
            torch.manual_seed(0); torch.set_num_threads(max(1, os.cpu_count() or 1))
        except ImportError:
            torch = None; print('[E7] torch not available; the sequence baseline is skipped')

        SEQ = 6                     # windows per sequence (15 s of context)

        def make_sequences(W, feats_, med, mu, sd):
            """Standardised feature matrix and, per window, the indices of the SEQ windows ending at it (same flight)."""
            X = ((W[feats_].to_numpy(dtype=float)) )
            X = np.where(np.isnan(X), med, X); X = (X - mu) / sd
            idx = np.zeros((len(W), SEQ), dtype=int)
            for fid, ii in W.groupby('flight_id', sort=False).indices.items():
                ii = ii[np.argsort(W.iloc[ii]['t_start'].to_numpy())]
                for k, i in enumerate(ii):
                    prev = ii[max(0, k - SEQ + 1):k + 1]
                    pad = np.full(SEQ - len(prev), prev[0]); idx[i] = np.r_[pad, prev]
            return X.astype(np.float32), idx

        if torch is not None:
            class SeqNet(nn.Module):
                def __init__(self, d):
                    super().__init__(); self.lstm = nn.LSTM(d, 64, batch_first=True); self.gru = nn.GRU(64, 32, batch_first=True); self.out = nn.Linear(32, 4)
                def forward(self, x):
                    h, _ = self.lstm(x); h, _ = self.gru(h); return self.out(h[:, -1])

        def train_seq(W_tr, feats_, seed, epochs):
            """Hybrid LSTM-GRU on sequences of window features (the architecture of reference [6]), class-weighted."""
            torch.manual_seed(seed); np.random.seed(seed)
            Xraw = W_tr[feats_].to_numpy(dtype=float)
            med = np.nanmedian(Xraw, 0); med = np.where(np.isnan(med), 0.0, med)
            Xf = np.where(np.isnan(Xraw), med, Xraw); mu, sd = Xf.mean(0), Xf.std(0) + 1e-6
            X, idx = make_sequences(W_tr, feats_, med, mu, sd)
            y = np.array([CLASSES.index(v) for v in W_tr['window_label']])
            cw = torch.tensor(len(y) / (4 * np.bincount(y, minlength=4) + 1e-9), dtype=torch.float32)
            net = SeqNet(X.shape[1]); opt = torch.optim.Adam(net.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=cw)
            Xt, It, yt = torch.tensor(X), torch.tensor(idx), torch.tensor(y)
            for ep in range(epochs):
                perm = torch.randperm(len(y))
                for b in range(0, len(y), 256):
                    j = perm[b:b + 256]; opt.zero_grad()
                    loss = lossf(net(Xt[It[j]]), yt[j]); loss.backward(); opt.step()
            net.eval()
            return {'net': net, 'med': med, 'mu': mu, 'sd': sd}

        def predict_seq(model, W, feats_):
            X, idx = make_sequences(W, feats_, model['med'], model['mu'], model['sd'])
            with torch.no_grad():
                out = []
                Xt, It = torch.tensor(X), torch.tensor(idx)
                for b in range(0, len(W), 2048):
                    out.append(torch.softmax(model['net'](Xt[It[b:b + 2048]]), 1).numpy())
            return np.vstack(out)

        def oof_seq(W_tr, feats_, seed, epochs, k=4):
            from sklearn.model_selection import StratifiedGroupKFold
            P = np.zeros((len(W_tr), 4)); fam = W_tr['family'].to_numpy(); grp = W_tr['flight_id'].to_numpy()
            for tr_i, te_i in StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(W_tr, fam, grp):
                m = train_seq(W_tr.iloc[tr_i], feats_, seed, epochs)
                P[te_i] = predict_seq(m, W_tr.iloc[te_i], feats_)
            return P

        def oneclass_fit(W_tr, feats_, seed):
            """Novelty detector trained on nominal windows only (the approach of reference [5]); flags windows whose
            isolation-forest score exceeds the 95th percentile of the nominal training windows."""
            from sklearn.ensemble import IsolationForest
            nom = W_tr[W_tr.window_label == 'nominal']
            Xraw = nom[feats_].to_numpy(dtype=float); med = np.nanmedian(Xraw, 0); med = np.where(np.isnan(med), 0.0, med)
            Xf = np.where(np.isnan(Xraw), med, Xraw); mu, sd = Xf.mean(0), Xf.std(0) + 1e-6
            iso = IsolationForest(n_estimators=300, random_state=seed).fit((Xf - mu) / sd)
            th = float(np.quantile(-iso.score_samples((Xf - mu) / sd), 0.95))
            return {'iso': iso, 'med': med, 'mu': mu, 'sd': sd, 'th': th}

        def oneclass_flags(model, W, feats_):
            X = W[feats_].to_numpy(dtype=float); X = np.where(np.isnan(X), model['med'], X); X = (X - model['mu']) / model['sd']
            return -model['iso'].score_samples(X) > model['th']

        def binary_metrics(flags, labels, tag):
            y = (np.asarray(labels) != 'nominal').astype(int); f = flags.astype(int)
            return {'tag': tag, 'acc': accuracy_score(y, f), 'macro_f1': f1_score(y, f, average='macro'),
                    'attack_recall': float(f[y == 1].mean()) if (y == 1).any() else np.nan, 'false_alarm': float(f[y == 0].mean()) if (y == 0).any() else np.nan}

        if torch is not None:
            try:
                _m = train_seq(Ws.iloc[:400], feats, 0, 1); _ = predict_seq(_m, Ws.iloc[:100], feats)
                print('[E7] sequence baseline smoke test passed')
            except Exception as ex:
                torch = None; print(f'[E7] sequence baseline disabled after smoke test failure: {type(ex).__name__}: {ex}')
        rows_leak, rows_flight, rows_loso = [], [], []
        # --- paired leakage audit for both architectures (same splits as E0)
        for rep_ in range(args.reps):
            rng = np.random.RandomState(500 + rep_)
            tr, ca, te, ex = split_flights(F, rng, n_train=30, n_cal=0, n_test=10 ** 6)
            W_tr, W_te = sub(Ws, tr), sub(Ws, te)
            idx = rng.permutation(len(Ws)); k = len(W_tr)
            W_rtr, W_rte = Ws.iloc[idx[:k]], Ws.iloc[idx[k:]]
            for tag, (A, Bt) in (('flight_grouped_split', (W_tr, W_te)), ('random_window_split', (W_rtr, W_rte))):
                if torch is not None:
                    m = train_seq(A, feats, rep_, args.e7_epochs); P = predict_seq(m, Bt, feats)
                    rows_leak.append({'rep': rep_, 'model': 'lstm_gru', **window_metrics(P, Bt['window_label'], tag)})
                oc = oneclass_fit(A, feats, rep_)
                rows_leak.append({'rep': rep_, 'model': 'one_class', **binary_metrics(oneclass_flags(oc, Bt, feats), Bt['window_label'], tag), 'ece': np.nan, 'nll': np.nan, 'brier': np.nan})
            print(f'[E7] leakage rep {rep_} done')
        # --- in-distribution flight level: the same stacking (probability summaries only) on the sequence model, and a
        #     declared flight rule for the one-class detector (flight flagged when more than 10 percent of windows are flagged)
        for rep_ in range(args.reps):
            rng = np.random.RandomState(100 + rep_)
            tr, ca, te, ex = split_flights(F, rng)
            W_tr, W_te = sub(Ws, tr), sub(Ws, te)
            if torch is not None:
                P_oof = oof_seq(W_tr, feats, rep_, args.e7_epochs)
                A_tr = flight_aggregates(W_tr, P_oof); cols = PROB_COLS(list(A_tr.columns.drop(['flight_id', 'family', 'subtype'])))
                fmod = fit(make_flight_model(rep_), A_tr[cols], A_tr['family'])
                m = train_seq(W_tr, feats, rep_, args.e7_epochs)
                A_te = flight_aggregates(W_te, predict_seq(m, W_te, feats)); S = A_te[['flight_id', 'family', 'subtype']].copy()
                Q = predict_proba(fmod, A_te[cols])
                for i, c in enumerate(CLASSES):
                    S[f'q_{c}'] = Q[:, i]
                S = finish_scores(S)
                rows_flight.append({'rep': rep_, 'model': 'lstm_gru_stacked', 'flight_acc': accuracy_score(S.family, S.pred),
                                    'flight_macro_f1': f1_score(S.family, S.pred, average='macro'), 'coarse_acc': accuracy_score(S.coarse_family, S.coarse_pred),
                                    'flight_ece10': flight_calibration(S)['ece10']})
            oc = oneclass_fit(W_tr, feats, rep_)
            fl = pd.DataFrame({'flight_id': W_te['flight_id'].to_numpy(), 'flag': oneclass_flags(oc, W_te, feats)}).groupby('flight_id').flag.mean()
            fam_te = W_te.groupby('flight_id').family.first()
            y = (fam_te != 'nominal').astype(int); f = (fl.reindex(fam_te.index) > 0.10).astype(int)
            rows_flight.append({'rep': rep_, 'model': 'one_class_flightrule', 'flight_acc': accuracy_score(y, f), 'flight_macro_f1': f1_score(y, f, average='macro'),
                                'coarse_acc': np.nan, 'flight_ece10': np.nan})
            print(f'[E7] in-distribution rep {rep_} done')
        # --- unseen subtype, single fit per hold-out (seed 0), same allocation rule as E2
        for fam in CLASSES[1:]:
            for st in sorted(F.loc[F.family == fam, 'subtype'].unique()):
                held = F.loc[F.subtype == st, 'flight_id'].tolist()
                rng = np.random.RandomState(700)
                seen = F.loc[(F.family == fam) & (F.subtype != st), 'flight_id'].tolist(); rng.shuffle(seen)
                n_tr = len(seen) // 3; n_ca = len(seen) // 3
                tr, ca, seen_test = seen[:n_tr], seen[n_tr:n_tr + n_ca], seen[n_tr + n_ca:]
                for c in CLASSES:
                    if c != fam:
                        ids = F.loc[F.family == c, 'flight_id'].tolist(); rng.shuffle(ids); tr += ids[:20]; ca += ids[20:40]
                W_tr = sub(Ws, tr); W_seen, W_held = sub(Ws, seen_test), sub(Ws, held)
                row = {'held_out_family': fam, 'held_out_subtype': st, 'n_unseen': len(held), 'n_seen_test': len(seen_test)}
                if torch is not None:
                    P_oof = oof_seq(W_tr, feats, 0, args.e7_epochs)
                    A_tr = flight_aggregates(W_tr, P_oof); cols = PROB_COLS(list(A_tr.columns.drop(['flight_id', 'family', 'subtype'])))
                    fmod = fit(make_flight_model(0), A_tr[cols], A_tr['family']); m = train_seq(W_tr, feats, 0, args.e7_epochs)
                    for tag_, W_ in (('unseen', W_held), ('seen', W_seen)):
                        A_ = flight_aggregates(W_, predict_seq(m, W_, feats)); S = A_[['flight_id', 'family', 'subtype']].copy()
                        Q = predict_proba(fmod, A_[cols])
                        for i, c in enumerate(CLASSES):
                            S[f'q_{c}'] = Q[:, i]
                        S = finish_scores(S)
                        row[f'lstm_gru_{tag_}_acc'] = float((S.pred == fam).mean()); row[f'lstm_gru_{tag_}_coarse_acc'] = float((S.coarse_pred == COARSE[fam]).mean())
                        row[f'lstm_gru_{tag_}_conf'] = float(S.conf.mean())
                oc = oneclass_fit(W_tr, feats, 0)
                for tag_, W_ in (('unseen', W_held), ('seen', W_seen)):
                    fl = pd.DataFrame({'flight_id': W_['flight_id'].to_numpy(), 'flag': oneclass_flags(oc, W_, feats)}).groupby('flight_id').flag.mean()
                    row[f'one_class_{tag_}_detected'] = float((fl > 0.10).mean())
                rows_loso.append(row); print(f'[E7] hold-out {st} done')
        pd.DataFrame(rows_leak).to_csv(out / 'e7_baselines_leakage.csv', index=False)
        pd.DataFrame(rows_flight).to_csv(out / 'e7_baselines_flight.csv', index=False)
        pd.DataFrame(rows_loso).to_csv(out / 'e7_baselines_loso.csv', index=False)
        if rows_leak:
            print('\n[E7] published architectures, paired leakage audit (window level):\n',
                  pd.DataFrame(rows_leak).groupby(['model', 'tag'])[['acc', 'macro_f1', 'ece']].mean().round(3).to_string())
        if rows_flight:
            print('\n[E7] published architectures, flight level in distribution:\n',
                  pd.DataFrame(rows_flight).groupby('model')[['flight_acc', 'flight_macro_f1', 'coarse_acc', 'flight_ece10']].agg(['mean', 'std']).round(3).to_string())
        if rows_loso:
            print('\n[E7] published architectures, unseen subtype (single fit):\n', pd.DataFrame(rows_loso).round(3).to_string(index=False))

    # ---------------- E3 Whelan case studies (stacked pipeline)
    if len(Ww) and 'e3' in RUN:
        tr, ca, te, ex = split_flights(F, np.random.RandomState(0), n_train=40, n_cal=20)
        W_tr, W_ca = sub(Ws, tr), sub(Ws, ca)
        wmodel, fmodel, agg_cols, T = stacked_pipeline(W_tr, feats, 0)
        P_ca = predict_proba(wmodel, W_ca[feats])
        S_cal = stacked_scores(W_ca, P_ca, fmodel, agg_cols)
        th10, th20 = conformal_thresholds(S_cal, 0.10), conformal_thresholds(S_cal, 0.20)
        thc10 = conformal_thresholds(S_cal, 0.10, 'coarse')
        Pw = predict_proba(wmodel, Ww[feats]); Pw_t = apply_temperature(Pw, T)
        Ww_out = Ww[['flight_id', 'family', 't_start', 't_end']].copy()
        for i, c in enumerate(CLASSES):
            Ww_out[f'p_{c}'] = Pw_t[:, i]
        Ww_out['pred'] = [CLASSES[i] for i in Pw_t.argmax(1)]
        Ww_out.to_csv(out / 'e3_whelan_window_timeline.csv', index=False)
        Sw = stacked_scores(Ww, Pw, fmodel, agg_cols)
        Sw['set_a0.10'] = [','.join(x) for x in conformal_sets(Sw, th10)]
        Sw['set_a0.20'] = [','.join(x) for x in conformal_sets(Sw, th20)]
        Sw['coarse_set_a0.10'] = [','.join(x) for x in conformal_sets(Sw, thc10, 'coarse')]
        Sw.to_csv(out / 'e3_whelan_flight_verdicts.csv', index=False)
        # receiver-derived disturbance intervals and a declared alert rule (case-study descriptives)
        ev = []
        for fid, d in Ww.groupby('flight_id', sort=False):
            d = d.sort_values('t_start'); idx = d.index.to_numpy()
            rx_dist = ((d['rx_fix_mean'] < 3) | (d['rx_sats_mean'] < 8)).to_numpy()
            p_non = 1.0 - Pw_t[idx, 0]
            alert = np.zeros(len(d), bool)
            for i in range(1, len(d)):
                if p_non[i] > 0.5 and p_non[i - 1] > 0.5:
                    alert[i - 1] = alert[i] = True
            t = d['t_start'].to_numpy()
            def intervals(mask):
                out_, start_ = [], None
                for i, v in enumerate(mask):
                    if v and start_ is None: start_ = t[i]
                    if (not v or i == len(mask) - 1) and start_ is not None:
                        out_.append((float(start_), float(t[i] + (5.0 if v else 0.0)))); start_ = None
                return out_
            dist_iv, alert_iv = intervals(rx_dist), intervals(alert)
            def delay_and_false(alert_iv_):
                delay_ = None
                if dist_iv and alert_iv_:
                    first = dist_iv[0][0]; later = [a for a in alert_iv_ if a[1] >= first]
                    delay_ = round(max(0.0, later[0][0] - first), 1) if later else None
                return delay_, sum(1 for a in alert_iv_ if not any(a[0] < e and a[1] > s_ for s_, e in dist_iv))
            delay, false_alerts = delay_and_false(alert_iv)
            cusum_starts = episode_starts_cusum(p_non, t)
            cusum_iv = [(float(x), float(x) + 5.0) for x in cusum_starts]
            cdelay, cfalse = delay_and_false(cusum_iv)
            ev.append({'flight_id': fid, 'receiver_disturbance_intervals_s': json.dumps(dist_iv), 'alert_intervals_s': json.dumps(alert_iv),
                       'detection_delay_s': delay, 'alerts_outside_disturbance': false_alerts,
                       'cusum_episode_starts_s': json.dumps([float(x) for x in cusum_starts]), 'cusum_detection_delay_s': cdelay,
                       'cusum_alerts_outside_disturbance': cfalse, 'n_windows': len(d)})
        pd.DataFrame(ev).to_csv(out / 'e3_whelan_events.csv', index=False)
        print('\n[E3] Whelan receiver-derived disturbance intervals; declared alert rule (P(non-nominal) > 0.5 for 2 consecutive windows) and CUSUM (k = 0.5, h = 0.75):\n',
              pd.DataFrame(ev).to_string(index=False))
        print('\n[E3] Whelan window predicted-class fractions per flight:\n',
              Ww_out.groupby('flight_id')['pred'].value_counts(normalize=True).round(2).to_string())
        print('\n[E3] Whelan flight verdicts (stacked) with conformal sets:\n',
              Sw[['flight_id', 'family', 'pred', 'conf', 'set_a0.10', 'set_a0.20', 'coarse_pred', 'coarse_set_a0.10']].round(3).to_string(index=False))
    print('\nresults written to', out)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/drive/MyDrive/UAV_GNSS_Research/uav-gnss-triage/src/sih_report.py
#!/usr/bin/env python3
"""
sih_report.py  --  manuscript tables and figures from a results folder written by sih_model.py.

Reads only committed result tables (reports/<run>/), so every number in the manuscript traces to a file in the
repository. Two optional inputs read raw data: --manifests (dataset composition table) and --whelan_dir (receiver
self-report channel table for the three live logs).

Usage:
  python sih_report.py --reports reports/v3 --figures figures/v3 \
      [--manifests data/sih/sih_flights_v2 data/sih/sih_flights_v2_nofix] [--whelan_dir data/features/features_v3/whelan_live]
Outputs: <reports>/tables/*.csv, <reports>/tables/tables.md, <figures>/*.png and *.pdf
"""
import argparse
import json
import pathlib

import numpy as np
import pandas as pd

CLASSES = ['nominal', 'spoof', 'gps_degrade', 'sensor_fault']
SUBTYPE_LABEL = {'jump': 'Jump', 'drift_incoherent': 'Incoherent drift', 'drift_coherent': 'Coherent drift', 'off_then_ok': 'Receiver silence',
                 'stuck_then_ok': 'Frozen receiver output', 'no_fix_then_ok': 'No-fix reporting', 'baro_stuck': 'Frozen barometer', 'mag_stuck': 'Frozen magnetometer'}
COARSE = ['nominal', 'gnss_chain', 'non_gnss_fault']
LABEL = {'nominal': 'Nominal', 'spoof': 'Spoof', 'gps_degrade': 'GNSS degradation', 'sensor_fault': 'Sensor fault',
         'gnss_chain': 'GNSS-chain inconsistency', 'non_gnss_fault': 'Non-GNSS fault',
         'rule': 'Percentile rule', 'stacked': 'Stacked', 'xgb': 'XGBoost', 'logreg': 'Logistic'}


def ms(x, digits=2):
    return f'{np.mean(x):.{digits}f} ± {np.std(x, ddof=0):.{digits}f}'


def md_table(df, floatfmt='.3f'):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(str(c) for c in cols) + ' |', '|' + '---|' * len(cols)]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            cells.append(f'{v:{floatfmt}}' if isinstance(v, (float, np.floating)) and not pd.isna(v) else str(v))
        lines.append('| ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--reports', required=True)
    ap.add_argument('--figures', required=True)
    ap.add_argument('--manifests', nargs='*', default=[])
    ap.add_argument('--whelan_dir', default=None)
    args = ap.parse_args()
    R, FIG = pathlib.Path(args.reports), pathlib.Path(args.figures)
    T = R / 'tables'; T.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
    md = []

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                         'figure.dpi': 120, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans'})

    def save(fig, name):
        fig.tight_layout()
        fig.savefig(FIG / f'{name}.png', bbox_inches='tight'); fig.savefig(FIG / f'{name}.pdf', bbox_inches='tight'); plt.close(fig)

    # ---------------------------------------------------------------- T1 dataset composition
    if args.manifests:
        rows = []
        for m in args.manifests:
            for line in open(pathlib.Path(m) / 'manifest.jsonl'):
                if line.strip():
                    r = json.loads(line)
                    if r.get('ok'):
                        rows.append({'run': pathlib.Path(m).name, 'family': r['family'], 'subtype': r['subtype'],
                                     'duration_s': r['verify'].get('duration_s'), 'firmware': r.get('firmware_commit', '')[:10],
                                     'onset_logged': r['verify'].get('onset_sim_s') is not None})
        D = pd.DataFrame(rows)
        t1 = D.groupby(['family', 'subtype']).agg(flights=('duration_s', 'size'), median_duration_s=('duration_s', 'median'),
                                                   min_duration_s=('duration_s', 'min'), max_duration_s=('duration_s', 'max')).reset_index()
        t1['family'] = t1['family'].map(LABEL)
        t1.to_csv(T / 't1_dataset_composition.csv', index=False)
        md.append('## Table 1. Generated dataset composition\n\n' + md_table(t1, '.0f') +
                  f'\n\nTotal flights: {len(D)}; firmware commits: {sorted(D.firmware.unique())}; '
                  f'onset logged for {int(D[D.family != "nominal"].onset_logged.sum())} of {int((D.family != "nominal").sum())} attacked flights.')

    # ---------------------------------------------------------------- T2 leakage audit
    e0 = pd.read_csv(R / 'e0_leakage_audit.csv')
    e0['tag'] = e0['tag'].map({'random_window_split': 'Random-window split', 'flight_grouped_split': 'Flight-grouped split'})
    g = e0.groupby('tag')
    t2 = pd.DataFrame({'Split': list(g.groups), 'Training windows': [int(v['n_train_windows'].mean()) for _, v in g],
                       'Accuracy': [ms(v['acc']) for _, v in g], 'Macro-F1': [ms(v['macro_f1']) for _, v in g],
                       'ECE': [ms(v['ece']) for _, v in g], 'NLL': [ms(v['nll']) for _, v in g], 'Brier': [ms(v['brier']) for _, v in g]})
    piv = e0.pivot(index='rep', columns='tag', values='acc'); d = piv['Random-window split'] - piv['Flight-grouped split']
    t2.to_csv(T / 't2_leakage_audit.csv', index=False)
    md.append('## Table 2. Leakage audit: same detector, matched training-window counts, paired over repetitions\n\n' + md_table(t2) +
              f'\n\nPaired accuracy difference (random-window minus flight-grouped): {d.mean():.3f} ± {d.std(ddof=0):.3f} over {len(d)} repetitions.')
    e0m = e0.groupby('tag')[['acc', 'macro_f1', 'ece']].mean().reset_index()
    fig, ax = plt.subplots(figsize=(4.2, 2.6))
    x = np.arange(3); w = 0.36
    for i, (_, r) in enumerate(e0m.iterrows()):
        ax.bar(x + (i - 0.5) * w, [r['acc'], r['macro_f1'], r['ece']], w, label=r['tag'])
    ax.set_xticks(x); ax.set_xticklabels(['Accuracy', 'Macro-F1', 'ECE']); ax.set_ylim(0, 1); ax.legend(frameon=False)
    ax.set_title('Window-level metrics by split protocol')
    save(fig, 'f_leakage_audit')

    # ---------------------------------------------------------------- T3 window metrics
    wm = pd.read_csv(R / 'e1_window_metrics.csv')
    g = wm.groupby(['model', 'tag'])
    t3 = pd.DataFrame({'Model': [LABEL[m] for m, _ in g.groups], 'Probabilities': [t for _, t in g.groups],
                       'Accuracy': [ms(v['acc']) for _, v in g], 'Macro-F1': [ms(v['macro_f1']) for _, v in g],
                       'ECE': [ms(v['ece']) for _, v in g], 'NLL': [ms(v['nll']) for _, v in g], 'Brier': [ms(v['brier']) for _, v in g]})
    t3['Probabilities'] = t3['Probabilities'].map({'raw': 'raw', 'temp_scaled': 'temperature-scaled'})
    t3.to_csv(T / 't3_window_metrics.csv', index=False)
    md.append('## Table 3. Window-level detector, mean ± std over 5 flight-grouped repetitions\n\n' + md_table(t3))

    # ---------------------------------------------------------------- T4 flight-level
    fm = pd.read_csv(R / 'e1_flight_metrics.csv')
    order = [a for a in ('rule', 'stacked_probs_only', 'stacked_raw_only', 'stacked', 'stacked_no_gap') if a in set(fm.aggregation)]
    LABEL.update({'stacked_probs_only': 'Stacked, probability summaries only', 'stacked_raw_only': 'Stacked, raw flight features only',
                  'stacked': 'Stacked, full', 'stacked_no_gap': 'Stacked, full, without derived-topic gap features'})
    rows = []
    for a in order:
        v = fm[fm.aggregation == a]
        rows.append({'Aggregation': LABEL[a], 'Flight accuracy': ms(v['flight_acc']), 'Flight macro-F1': ms(v['flight_macro_f1']), 'AURC': ms(v['aurc']),
                     'Coarse accuracy': ms(v['coarse_acc']), 'Coarse macro-F1': ms(v['coarse_macro_f1'])})
    t4 = pd.DataFrame(rows); t4.to_csv(T / 't4_flight_level.csv', index=False)
    md.append('## Table 4. Flight-level verdicts, mean ± std over 5 repetitions (20 train / 20 calibration / 20 test flights per family)\n\n' + md_table(t4))
    rows = []
    for a in order:
        v = fm[fm.aggregation == a]
        rows.append({'Aggregation': LABEL[a], 'Fine ECE (10 bins)': ms(v['flight_ece10']), 'Fine NLL': ms(v['flight_nll']), 'Fine Brier': ms(v['flight_brier']),
                     'Coarse ECE (10 bins)': ms(v['coarse_ece10']), 'Coarse NLL': ms(v['coarse_nll']), 'Coarse Brier': ms(v['coarse_brier'])})
    t4b = pd.DataFrame(rows); t4b.to_csv(T / 't4b_flight_calibration.csv', index=False)
    md.append('## Table 4b. Flight-level probability calibration of the verdict probabilities (test flights), mean ± std over 5 repetitions\n\n' + md_table(t4b))
    # reliability diagram from pooled test flights (all repetitions), full stacked and rule
    fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.0), sharey=True)
    for ax, a in zip(axes, ('rule', 'stacked')):
        files = sorted(R.glob(f'e1_rep*_{a}_flight_scores.csv'))
        if not files:
            continue
        S = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
        conf = S['conf'].to_numpy(); correct = (S['pred'] == S['family']).to_numpy().astype(float)
        edges = np.linspace(0, 1, 11); mids, accs, cnts = [], [], []
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.sum():
                mids.append(conf[m].mean()); accs.append(correct[m].mean()); cnts.append(int(m.sum()))
        ax.plot([0, 1], [0, 1], 'k--', lw=0.8); ax.plot(mids, accs, 'o-', ms=4)
        for x_, y_, n_ in zip(mids, accs, cnts):
            ax.annotate(str(n_), (x_, y_), textcoords='offset points', xytext=(0, 5), ha='center', fontsize=6)
        ax.set_title(LABEL[a]); ax.set_xlabel('Flight confidence'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    axes[0].set_ylabel('Fraction correct')
    save(fig, 'f_flight_reliability')

    # ---------------------------------------------------------------- T5 conformal
    cf = pd.read_csv(R / 'e1_conformal.csv')
    cols5 = ['coverage', 'avg_set_size', 'multi_label_rate', 'empty_rate', 'non_singleton_rate', 'verdict_outside_set_rate', 'operational_abstain_rate']
    gm = cf.groupby(['aggregation', 'level', 'alpha', 'family'])[cols5]
    agg = gm.mean().reset_index(); sd = gm.std(ddof=0).reset_index()
    t5 = agg[['aggregation', 'level', 'alpha', 'family']].copy()
    for c in cols5:
        t5[c] = [f'{m_:.2f} ± {s_:.2f}' for m_, s_ in zip(agg[c], sd[c])]
    t5['aggregation'] = t5['aggregation'].map(LABEL); t5['family'] = t5['family'].map(LABEL)
    t5 = t5.rename(columns={'aggregation': 'Aggregation', 'level': 'Level', 'family': 'Class', 'coverage': 'Coverage', 'avg_set_size': 'Mean set size',
                            'multi_label_rate': 'Multi-label', 'empty_rate': 'Empty', 'non_singleton_rate': 'Non-singleton', 'verdict_outside_set_rate': 'Verdict outside set',
                            'operational_abstain_rate': 'Operational abstention'})
    t5.to_csv(T / 't5_conformal.csv', index=False)
    md.append('## Table 5. Class-conditional conformal prediction sets at the flight level, mean ± std over 5 repetitions (balanced test sets)\n\n' + md_table(t5))
    agg['aggregation'] = agg['aggregation'].map(LABEL); agg['family'] = agg['family'].map(LABEL)
    sub = agg[(agg.aggregation == LABEL['stacked']) & (agg.alpha == 0.1)]
    sub = sub.assign(abstain_rate=sub['operational_abstain_rate'])
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0), sharey=True, gridspec_kw={'wspace': 0.12, 'width_ratios': [4, 3]})
    wrap = {'GNSS degradation': 'GNSS\ndegradation', 'GNSS-chain inconsistency': 'GNSS-chain\ninconsistency', 'Non-GNSS fault': 'Non-GNSS\nfault', 'Sensor fault': 'Sensor\nfault'}
    for ax, lvl, classes in ((axes[0], 'fine', CLASSES), (axes[1], 'coarse', COARSE)):
        d = sub[sub.level == lvl].set_index('family').reindex([LABEL[c] for c in classes])
        x = np.arange(len(d)); w = 0.38
        ax.bar(x - w / 2, d['coverage'], w, label='Coverage'); ax.bar(x + w / 2, d['abstain_rate'], w, label='Operational abstention')
        ax.axhline(0.9, ls='--', lw=0.8, color='k', label='Target coverage 0.90'); ax.set_xticks(x)
        ax.set_xticklabels([wrap.get(n, n) for n in d.index], fontsize=8)
        ax.set_title(f'{lvl.capitalize()} level, alpha = 0.10'); ax.set_ylim(0, 1.05)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, frameon=False, ncol=3, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, 1.04))
    save(fig, 'f_conformal_stacked')

    per_rep = cf[(cf.aggregation == 'stacked') & (cf.alpha == 0.1) & (cf.level == 'fine')].pivot(index='rep', columns='family', values='coverage')
    per_rep = per_rep[[c for c in CLASSES if c in per_rep.columns]].rename(columns=LABEL).reset_index().rename(columns={'rep': 'Repetition'})
    per_rep.to_csv(T / 't5c_coverage_per_repetition.csv', index=False)
    md.append('## Table 5c. Fine-level coverage at alpha = 0.10 for the full stacked model, per repetition (20 test flights per class)\n\n' + md_table(per_rep, '.2f'))
    if (R / 'e1_extra_flights.csv').exists():
        ex = pd.read_csv(R / 'e1_extra_flights.csv')
        exm = ex.groupby(['aggregation', 'level', 'alpha', 'family'])[['n', 'acc', 'coverage', 'operational_abstain_rate']].mean().reset_index()
        exm['aggregation'] = exm['aggregation'].map(LABEL); exm['family'] = exm['family'].map(LABEL)
        t5b = exm.rename(columns={'aggregation': 'Aggregation', 'level': 'Level', 'family': 'Class', 'n': 'Flights per rep', 'acc': 'Accuracy',
                                  'coverage': 'Coverage', 'operational_abstain_rate': 'Operational abstention'})
        t5b.to_csv(T / 't5b_extra_flights.csv', index=False)
        md.append('## Table 5b. Flights beyond the balanced test set (GNSS degradation), scored with the same thresholds, mean over 5 repetitions\n\n' + md_table(t5b))

    # ---------------------------------------------------------------- F risk-coverage
    rc = pd.read_csv(R / 'e1_risk_coverage.csv')
    fig, ax = plt.subplots(figsize=(4.0, 3.0))
    grid = np.linspace(0.05, 1.0, 96)
    for a in ('rule', 'stacked'):
        if a not in set(rc.aggregation):
            continue
        curves = []
        for rep, d in rc[rc.aggregation == a].groupby('rep'):
            d = d.sort_values('coverage')
            curves.append(np.interp(grid, d['coverage'], d['risk']))
        c = np.array(curves)
        ax.plot(grid, c.mean(0), label=LABEL[a]); ax.fill_between(grid, c.min(0), c.max(0), alpha=0.2)
    ax.set_xlabel('Coverage (fraction of flights given a verdict)'); ax.set_ylabel('Selective risk (error rate)')
    ax.set_title('Risk-coverage, flight level'); ax.legend(frameon=False)
    save(fig, 'f_risk_coverage')

    # ---------------------------------------------------------------- T6 leave-one-subtype-out (single-fit design)
    lo = pd.read_csv(R / 'e2_leave_one_subtype_out.csv')
    ps = pd.read_csv(R / 'e2_per_seed.csv') if (R / 'e2_per_seed.csv').exists() else None
    n_seeds = int(ps['seed'].nunique()) if ps is not None else 1
    pm = lambda a, b: [f'{m_:.2f} ± {s_:.2f}' for m_, s_ in zip(lo[a], lo[b])]
    t6 = pd.DataFrame({'Held-out family': lo['held_out_family'].map(LABEL), 'Held-out subtype': lo['held_out_subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v)),
                       'n unseen': lo['unseen_n_mean'].round(0).astype(int), 'n seen test': lo['seen_n_mean'].round(0).astype(int),
                       'Fine accuracy, unseen': pm('unseen_acc_mean', 'unseen_acc_std'), 'Fine accuracy, seen': pm('seen_acc_mean', 'seen_acc_std'),
                       'Coarse accuracy, unseen': pm('unseen_coarse_acc_mean', 'unseen_coarse_acc_std'), 'Coarse accuracy, seen': pm('seen_coarse_acc_mean', 'seen_coarse_acc_std'),
                       'Mean confidence, unseen': pm('unseen_mean_conf_mean', 'unseen_mean_conf_std'), 'Mean confidence, seen': pm('seen_mean_conf_mean', 'seen_mean_conf_std')})
    t6.to_csv(T / 't6_leave_one_subtype_out.csv', index=False)
    md.append(f'## Table 6. Unseen-subtype evaluation: one fitted pipeline per hold-out (single training and calibration set) evaluated on the withheld subtype and on a '
              f'disjoint seen-subtype test set from the same family, mean ± std over {n_seeds} seeds\n\n' + md_table(t6))
    t6b = pd.DataFrame({'Held-out subtype': lo['held_out_subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v)),
                        'Fine coverage (0.10), unseen': pm('unseen_coverage_a0.1_mean', 'unseen_coverage_a0.1_std'), 'Fine coverage (0.10), seen': pm('seen_coverage_a0.1_mean', 'seen_coverage_a0.1_std'),
                        'Coarse coverage (0.10), unseen': pm('unseen_coarse_coverage_a0.1_mean', 'unseen_coarse_coverage_a0.1_std'), 'Coarse coverage (0.10), seen': pm('seen_coarse_coverage_a0.1_mean', 'seen_coarse_coverage_a0.1_std'),
                        'Operational abstention (0.10), unseen': pm('unseen_operational_abstain_a0.1_mean', 'unseen_operational_abstain_a0.1_std'), 'Operational abstention (0.10), seen': pm('seen_operational_abstain_a0.1_mean', 'seen_operational_abstain_a0.1_std'),
                        'Predicted as (unseen, seed 0)': lo['unseen_pred_dist_seed0']})
    t6b.to_csv(T / 't6b_unseen_vs_seen_coverage.csv', index=False)
    md.append('## Table 6b. Same fitted pipelines: conformal coverage and operational abstention on the withheld subtype and on the seen-subtype test set\n\n' + md_table(t6b))
    if (R / 'e2_splits.csv').exists():
        sp = pd.read_csv(R / 'e2_splits.csv'); sp = sp[sp.seed == 0].drop(columns=['seed']); sp['family'] = sp['family'].map(LABEL)
        sp['held_out_subtype'] = sp['held_out_subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v))
        t6c = sp.rename(columns={'held_out_subtype': 'Held-out subtype', 'family': 'Family', 'n_train': 'Train', 'n_cal': 'Calibration', 'n_seen_test': 'Seen test',
                                 'rank_alpha0.1': 'Rank (0.10)', 'rank_alpha0.2': 'Rank (0.20)', 'rank_at_max_alpha0.1': 'Threshold at max score (0.10)'})
        t6c.to_csv(T / 't6c_holdout_allocation.csv', index=False)
        md.append('## Table 6c. Hold-out allocation per family (identical across seeds), test-set sizes, and the conformal order-statistic ranks; where the rank equals the '
                  'calibration size the threshold is the largest calibration score\n\n' + md_table(t6c, '.0f'))
    if 'unseen_gate_mahal_withheld_mean' in lo.columns:
        t6d = pd.DataFrame({'Held-out subtype': lo['held_out_subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v)),
                            'n unseen': lo['unseen_n_mean'].round(0).astype(int), 'n seen test': lo['seen_n_mean'].round(0).astype(int),
                            'Mahalanobis gate, unseen': pm('unseen_gate_mahal_withheld_mean', 'unseen_gate_mahal_withheld_std'),
                            'Mahalanobis gate, seen': pm('seen_gate_mahal_withheld_mean', 'seen_gate_mahal_withheld_std'),
                            'Isolation-forest gate, unseen': pm('unseen_gate_iso_withheld_mean', 'unseen_gate_iso_withheld_std'),
                            'Isolation-forest gate, seen': pm('seen_gate_iso_withheld_mean', 'seen_gate_iso_withheld_std'),
                            **({'Gap-rule baseline, unseen': pm('unseen_gate_gaprule_withheld_mean', 'unseen_gate_gaprule_withheld_std'),
                                'Gap-rule baseline, seen': pm('seen_gate_gaprule_withheld_mean', 'seen_gate_gaprule_withheld_std')}
                               if 'unseen_gate_gaprule_withheld_mean' in lo.columns else {}),
                            'Conformal policy alone, unseen': pm('unseen_operational_abstain_a0.1_mean', 'unseen_operational_abstain_a0.1_std'),
                            'Conformal or Mahalanobis gate, unseen': pm('unseen_gate_mahal_or_conformal_withheld_mean', 'unseen_gate_mahal_or_conformal_withheld_std'),
                            'Conformal or Mahalanobis gate, seen': pm('seen_gate_mahal_or_conformal_withheld_mean', 'seen_gate_mahal_or_conformal_withheld_std')})
        t6d.to_csv(T / 't6d_distributional_gate.csv', index=False)
        if 'unseen_gate_mahal_auroc_mean' in lo.columns:
            t6e = pd.DataFrame({'Held-out subtype': lo['held_out_subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v)),
                                'Mahalanobis AUROC': pm('unseen_gate_mahal_auroc_mean', 'unseen_gate_mahal_auroc_std'),
                                'Mahalanobis catch at 5% FPR': pm('unseen_gate_mahal_catch_at_5pct_fpr_mean', 'unseen_gate_mahal_catch_at_5pct_fpr_std'),
                                'Isolation-forest AUROC': pm('unseen_gate_iso_auroc_mean', 'unseen_gate_iso_auroc_std'),
                                'Isolation-forest catch at 5% FPR': pm('unseen_gate_iso_catch_at_5pct_fpr_mean', 'unseen_gate_iso_catch_at_5pct_fpr_std'),
                                'Plain confidence AUROC': pm('unseen_gate_one_minus_conf_auroc_mean', 'unseen_gate_one_minus_conf_auroc_std'),
                                'Plain confidence catch at 5% FPR': pm('unseen_gate_one_minus_conf_catch_at_5pct_fpr_mean', 'unseen_gate_one_minus_conf_catch_at_5pct_fpr_std'),
                                **({'Gap-rule AUROC': pm('unseen_gate_gaprule_auroc_mean', 'unseen_gate_gaprule_auroc_std'),
                                    'Gap-rule catch at 5% FPR': pm('unseen_gate_gaprule_catch_at_5pct_fpr_mean', 'unseen_gate_gaprule_catch_at_5pct_fpr_std')}
                                   if 'unseen_gate_gaprule_auroc_mean' in lo.columns else {})})
            t6e.to_csv(T / 't6e_gate_separability.csv', index=False)
            md.append('## Table 6e. Threshold-free separability of the withheld subtype from the seen-subtype test set: AUROC of each novelty score (and of one minus '
                      'confidence), and the fraction of withheld flights caught at the score cut that withholds 5 percent of seen flights\n\n' + md_table(t6e))
        md.append('## Table 6d. Distributional abstention gate: fraction of flights withheld on the withheld subtype and on the seen-subtype test set (thresholds at the '
                  '95th percentile of the calibration flights), and the combined policy\n\n' + md_table(t6d))
        fig, ax = plt.subplots(figsize=(7.2, 3.2))
        x = np.arange(len(lo)); w = 0.26
        ax.bar(x - w, lo['unseen_operational_abstain_a0.1_mean'], w, yerr=lo['unseen_operational_abstain_a0.1_std'], label='Conformal policy alone (mean ± std, 3 seeds)', capsize=2)
        ax.bar(x, lo['unseen_gate_mahal_withheld_mean'], w, yerr=lo['unseen_gate_mahal_withheld_std'], label='Mahalanobis gate', capsize=2)
        ax.bar(x + w, lo['unseen_gate_mahal_or_conformal_withheld_mean'], w, yerr=lo['unseen_gate_mahal_or_conformal_withheld_std'], label='Combined', capsize=2)
        ax.plot(x, lo['seen_gate_mahal_or_conformal_withheld_mean'], 'kv', ms=5, label='Combined, seen-subtype test set')
        ax.set_xticks(x); ax.set_xticklabels([SUBTYPE_LABEL.get(v, v) for v in lo['held_out_subtype']], rotation=25, ha='right', fontsize=7); ax.set_ylim(0, 1.05)
        ax.set_ylabel('Fraction withheld'); ax.set_title('Withheld subtype: what each abstention mechanism catches', pad=24)
        ax.legend(frameon=False, ncol=4, fontsize=7, loc='lower center', bbox_to_anchor=(0.5, 1.0))
        save(fig, 'f_distributional_gate')
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    x = np.arange(len(lo)); w = 0.2
    ax.bar(x - 1.5 * w, lo['unseen_acc_mean'], w, yerr=lo['unseen_acc_std'], label='Fine accuracy, unseen (mean ± std, 3 seeds)', capsize=2)
    ax.bar(x - 0.5 * w, lo['seen_acc_mean'], w, yerr=lo['seen_acc_std'], label='Fine accuracy, seen test (mean ± std, 3 seeds)', capsize=2)
    ax.bar(x + 0.5 * w, lo['unseen_coverage_a0.1_mean'], w, yerr=lo['unseen_coverage_a0.1_std'], label='Fine coverage (0.10), unseen', capsize=2)
    ax.bar(x + 1.5 * w, lo['seen_coverage_a0.1_mean'], w, yerr=lo['seen_coverage_a0.1_std'], label='Fine coverage (0.10), seen test', capsize=2)
    ax.plot(x, lo['unseen_mean_conf_mean'], 'k_', ms=14, mew=2, label='Mean confidence, unseen')
    ax.axhline(0.9, ls='--', lw=0.8, color='k')
    ax.set_xticks(x); ax.set_xticklabels([SUBTYPE_LABEL.get(v, v) for v in lo['held_out_subtype']], rotation=25, ha='right', fontsize=7); ax.set_ylim(0, 1.05)
    ax.set_title('Same fitted pipeline on the withheld subtype and on a seen-subtype test set', pad=28)
    ax.legend(frameon=False, ncol=3, fontsize=7, loc='lower center', bbox_to_anchor=(0.5, 1.0))
    save(fig, 'f_unseen_subtype_collapse')

    # ---------------------------------------------------------------- T7 Whelan
    wv = pd.read_csv(R / 'e3_whelan_flight_verdicts.csv')
    wv['case'] = wv['flight_id'].str.extract(r'ace-(\w+?)-')[0].str.capitalize()
    t7 = wv[['case', 'family', 'pred', 'conf', 'set_a0.10', 'set_a0.20', 'coarse_pred', 'coarse_set_a0.10']].copy()
    for c in ('set_a0.10', 'set_a0.20', 'coarse_set_a0.10'):
        t7[c] = t7[c].fillna('(empty)').replace('', '(empty)')
    t7['family'] = t7['family'].map(LABEL); t7['pred'] = t7['pred'].map(LABEL); t7['coarse_pred'] = t7['coarse_pred'].map(LABEL)
    t7 = t7.rename(columns={'case': 'Log', 'family': 'Reference label', 'pred': 'Verdict', 'conf': 'Confidence',
                            'set_a0.10': 'Set (0.10)', 'set_a0.20': 'Set (0.20)', 'coarse_pred': 'Coarse verdict', 'coarse_set_a0.10': 'Coarse set (0.10)'})
    t7.to_csv(T / 't7_whelan_verdicts.csv', index=False)
    md.append('## Table 7. Whelan live logs: physics-channel verdicts (detector trained on simulation only)\n\n' + md_table(t7))
    if args.whelan_dir:
        try:
            from pyulog import ULog
            rows = []
            for f in sorted(pathlib.Path(args.whelan_dir).glob('*.ulg')):
                u = ULog(str(f)); g = next(d for d in u.data_list if d.name in ('vehicle_gps_position', 'sensor_gps'))
                lat = np.asarray(g.data.get('lat', g.data.get('latitude_deg')), float); lon = np.asarray(g.data.get('lon', g.data.get('longitude_deg')), float)
                if np.nanmax(np.abs(lat)) > 1000:
                    lat, lon = lat / 1e7, lon / 1e7
                jump = float(np.max(np.hypot(np.diff(lat) * 111320, np.diff(lon) * 111320 * np.cos(np.radians(lat[:-1])))))
                rows.append({'Log': f.name.split('-')[1].capitalize(), 'fix_type min': int(np.min(g.data['fix_type'])),
                             'satellites min': int(np.min(g.data['satellites_used'])), 'satellites median': int(np.median(g.data['satellites_used'])),
                             'noise max': int(np.max(g.data['noise_per_ms'])), 'jamming indicator max': int(np.max(g.data['jamming_indicator'])),
                             'eph max (m)': float(np.max(g.data['eph'])), 'speed variance median': float(np.median(g.data['s_variance_m_s'])),
                             'speed variance max': float(np.max(g.data['s_variance_m_s'])), 'max fix-to-fix jump (m)': jump,
                             'lat range (deg)': f'{lat.min():.5f} to {lat.max():.5f}', 'lon range (deg)': f'{lon.min():.5f} to {lon.max():.5f}'})
            t7b = pd.DataFrame(rows); t7b.to_csv(T / 't7b_whelan_receiver_channel.csv', index=False)
            md.append('## Table 7b. Whelan live logs: receiver self-report channel\n\n' + md_table(t7b, '.2f'))
        except Exception as e:
            md.append(f'(receiver-channel table skipped: {type(e).__name__}: {e})')
    tl = pd.read_csv(R / 'e3_whelan_window_timeline.csv')
    events = pd.read_csv(R / 'e3_whelan_events.csv').set_index('flight_id') if (R / 'e3_whelan_events.csv').exists() else None
    if events is not None:
        t7c = events.reset_index().copy(); t7c['flight_id'] = t7c['flight_id'].str.extract(r'ace-(\w+?)-')[0].str.capitalize()
        t7c = t7c.rename(columns={'flight_id': 'Log', 'receiver_disturbance_intervals_s': 'Receiver disturbance intervals (s)', 'alert_intervals_s': 'Alert intervals (s)',
                                  'detection_delay_s': 'Detection delay (s)', 'alerts_outside_disturbance': 'Alerts outside disturbance', 'n_windows': 'Windows',
                                  'cusum_episode_starts_s': 'CUSUM episode starts (s)', 'cusum_detection_delay_s': 'CUSUM detection delay (s)',
                                  'cusum_alerts_outside_disturbance': 'CUSUM alerts outside disturbance'})
        t7c.to_csv(T / 't7c_whelan_events.csv', index=False)
        md.append('## Table 7c. Live logs: receiver-derived disturbance intervals (fix type below 3 or fewer than 8 satellites) and alerts from the declared rule '
                  '(P(non-nominal) above 0.5 in two consecutive windows)\n\n' + md_table(t7c, '.1f'))
    fig, axes = plt.subplots(3, 1, figsize=(6.4, 5.0), sharex=False)
    for ax, (fid, d) in zip(axes, tl.groupby('flight_id', sort=False)):
        d = d.sort_values('t_start')
        ax.stackplot(d['t_start'], [d[f'p_{c}'] for c in CLASSES], labels=[LABEL[c] for c in CLASSES], alpha=0.85)
        if events is not None and fid in events.index:
            for s_, e_ in json.loads(events.loc[fid, 'receiver_disturbance_intervals_s']):
                ax.axvspan(s_, e_, color='k', alpha=0.15, lw=0)
        ax.set_ylim(0, 1); ax.set_ylabel('P(class)'); ax.set_title(fid.replace('whelan_', '').split('_2033')[0], fontsize=8)
    axes[-1].set_xlabel('Time since log start (s)'); axes[0].legend(frameon=False, ncol=4, fontsize=7, loc='lower left')
    axes[-1].text(0.99, 0.02, 'shaded: receiver-derived disturbance', transform=axes[-1].transAxes, ha='right', fontsize=6)
    save(fig, 'f_whelan_timelines')

    # ---------------------------------------------------------------- T7d onset localisation
    if (R / 'e5_onset_summary.csv').exists():
        e5 = pd.read_csv(R / 'e5_onset_summary.csv')
        e5['family'] = e5['family'].map(LABEL); e5['subtype'] = e5['subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v if v != 'none' else ''))
        if 'method' in e5.columns:
            e5['method'] = e5['method'].map({'rule': 'Declared rule', 'cusum': 'CUSUM'})
            e5 = e5.sort_values(['method', 'family', 'subtype'], ascending=[False, True, True])
        t7d = e5.rename(columns={'method': 'Method', 'family': 'Family', 'subtype': 'Subtype', 'n': 'Flights', 'detected_within_60s': 'Detected within 60 s',
                                 'median_abs_error_s': 'Median |onset error| (s)', 'within_10s': 'Within 10 s', 'within_30s': 'Within 30 s', 'median_signed_error_s': 'Median signed error (s)',
                                 'early_alert_rate': 'Early or false alert rate'})
        t7d.to_csv(T / 't7d_onset_localisation.csv', index=False)
        md.append('## Table 7d. Onset localisation on held-out flights with the declared alert rule against the logged onset (one fitted pipeline); for nominal '
                  'flights the last column is the false-alert rate\n\n' + md_table(t7d, '.2f'))
        det = pd.read_csv(R / 'e5_onset_localisation.csv')
        if 'method' in det.columns:
            det = det[det.method == 'rule']
        att = det[(det.family != 'nominal') & det.onset_error_s.notna()]
        if len(att):
            fig, ax = plt.subplots(figsize=(6.4, 3.0))
            order = [st for st in SUBTYPE_LABEL if st in set(att.subtype)]
            data = [att.loc[att.subtype == st, 'onset_error_s'].to_numpy() for st in order]
            ax.boxplot(data, tick_labels=[SUBTYPE_LABEL[st] for st in order], showfliers=True, whis=(5, 95))
            ax.axhline(0, ls='--', lw=0.8, color='k'); ax.set_ylabel('Estimated minus logged onset (s)')
            ax.set_title('Onset localisation error on held-out flights'); plt.setp(ax.get_xticklabels(), rotation=25, ha='right', fontsize=7)
            save(fig, 'f_onset_localisation')

    # ---------------------------------------------------------------- T10 real PX4 logs
    if (R / 'e6_real_summary.csv').exists():
        e6 = pd.read_csv(R / 'e6_real_summary.csv').iloc[0]
        t10 = pd.DataFrame({'Quantity': ['Flights (after removing duplicate uploads)', 'Windows', 'Verdict: nominal', 'Verdict: spoof', 'Verdict: GNSS degradation', 'Verdict: sensor fault',
                                         'Coarse verdict: nominal', 'Coarse verdict: GNSS-chain inconsistency', 'Coarse verdict: non-GNSS fault',
                                         'Operational abstention, fine (0.10)', 'Operational abstention, coarse (0.10)', 'Mahalanobis gate withheld',
                                         'Isolation-forest gate withheld', 'Gap-rule baseline withheld', 'Mean confidence', 'Median confidence',
                                         'Flights with at least one alert episode', 'Nominal verdict, not withheld by set or gate',
                                         'Missing base features per flight, mean', 'Missing base features per flight, max'],
                            'Value': [int(e6['n_flights']), int(e6['n_windows']), e6['verdict_frac_nominal'], e6['verdict_frac_spoof'], e6['verdict_frac_gps_degrade'],
                                      e6['verdict_frac_sensor_fault'], e6['coarse_verdict_frac_nominal'], e6['coarse_verdict_frac_gnss_chain'], e6['coarse_verdict_frac_non_gnss_fault'],
                                      e6['abstain_fine_rate'], e6['abstain_coarse_rate'], e6['gate_mahal_withheld'], e6['gate_iso_withheld'], e6['gate_gaprule_withheld'],
                                      e6['mean_conf'], e6['median_conf'], e6['frac_with_alert_episode'], e6['nominal_and_not_withheld_frac'],
                                      e6.get('missing_frac_mean', float('nan')), e6.get('missing_frac_max', float('nan'))]})
        t10.to_csv(T / 't10_real_logs.csv', index=False)
        md.append('## Table 10. Real PX4 flight logs without ground truth: verdict distribution, abstention and gate behaviour of the pipeline trained on simulation only\n\n' + md_table(t10))
        if 'n_low_missing' in e6.index:
            t10c = pd.DataFrame({'Missing base features': ['5 percent or less', 'more than 5 percent'],
                                 'Flights': [int(e6['n_low_missing']), int(e6['n_high_missing'])],
                                 'Nominal verdict': [e6['nominal_low_missing'], e6['nominal_high_missing']],
                                 'Operational abstention': [e6['abstain_low_missing'], e6['abstain_high_missing']],
                                 'Mahalanobis gate withheld': [e6['gate_withheld_low_missing'], e6['gate_withheld_high_missing']]})
            t10c.to_csv(T / 't10c_real_logs_by_missingness.csv', index=False)
            md.append('## Table 10c. Real PX4 logs split by the fraction of missing base features per flight\n\n' + md_table(t10c))
        if (R / 'e6_real_by_version.csv').exists():
            bv = pd.read_csv(R / 'e6_real_by_version.csv').rename(columns={'ver_sw_release': 'Firmware release', 'px4_version': 'PX4 release', 'n': 'Flights', 'nominal': 'Nominal verdict',
                                                                          'abstain': 'Operational abstention', 'gate': 'Mahalanobis gate withheld', 'conf': 'Mean confidence'})
            bv.to_csv(T / 't10b_real_logs_by_version.csv', index=False)
            md.append('## Table 10b. Real PX4 logs by firmware release\n\n' + md_table(bv))
        vr = pd.read_csv(R / 'e6_real_flight_verdicts.csv')
        fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))
        axes[0].hist(vr['conf'], bins=np.linspace(0, 1, 21)); axes[0].set_xlabel('Flight confidence'); axes[0].set_ylabel('Flights'); axes[0].set_title('Real logs: confidence')
        counts = vr['pred'].value_counts().reindex(CLASSES).fillna(0)
        axes[1].bar([LABEL[c] for c in CLASSES], counts.to_numpy()); axes[1].set_title('Real logs: verdicts'); plt.setp(axes[1].get_xticklabels(), rotation=20, ha='right', fontsize=7)
        save(fig, 'f_real_logs')

    # ---------------------------------------------------------------- T16 published architectures under the same protocol
    if (R / 'e7_baselines_leakage.csv').exists():
        lk = pd.read_csv(R / 'e7_baselines_leakage.csv')
        e0b = pd.read_csv(R / 'e0_leakage_audit.csv'); e0b['model'] = 'xgboost'
        allk = pd.concat([e0b[['rep', 'model', 'tag', 'acc', 'macro_f1', 'ece']], lk[['rep', 'model', 'tag', 'acc', 'macro_f1', 'ece']]], ignore_index=True)
        MODEL = {'xgboost': 'XGBoost (this paper)', 'lstm_gru': 'LSTM-GRU sequence model [6]', 'one_class': 'One-class novelty detector [5], nominal vs attack'}
        rows = []
        for m_ in ('xgboost', 'lstm_gru', 'one_class'):
            d = allk[allk.model == m_]
            if not len(d):
                continue
            g_ = d.groupby('tag'); rnd = g_.get_group('random_window_split') if 'random_window_split' in g_.groups else None; grp = g_.get_group('flight_grouped_split')
            piv = d.pivot(index='rep', columns='tag', values='acc'); diff = piv['random_window_split'] - piv['flight_grouped_split']
            rows.append({'Detector': MODEL[m_], 'Random-window accuracy': ms(rnd['acc']) if rnd is not None else '', 'Flight-grouped accuracy': ms(grp['acc']),
                         'Paired difference': ms(diff), 'Random-window macro-F1': ms(rnd['macro_f1']), 'Flight-grouped macro-F1': ms(grp['macro_f1']),
                         'Flight-grouped ECE': ms(grp['ece']) if grp['ece'].notna().any() else 'n/a'})
        t16 = pd.DataFrame(rows); t16.to_csv(T / 't16_published_architectures_leakage.csv', index=False)
        md.append('## Table 16. Published architectures under the same paired leakage audit (window level; the one-class detector is scored on nominal versus attack)\n\n' + md_table(t16))
        if (R / 'e7_baselines_flight.csv').exists():
            fb = pd.read_csv(R / 'e7_baselines_flight.csv'); g_ = fb.groupby('model')
            FM = {'lstm_gru_stacked': 'LSTM-GRU with the same flight-level stacking', 'one_class_flightrule': 'One-class detector, flight flagged above 10 percent of windows (nominal vs attack)'}
            t16b = pd.DataFrame({'Detector': [FM.get(k, k) for k in g_.groups], 'Flight accuracy': [ms(v['flight_acc']) for _, v in g_],
                                 'Flight macro-F1': [ms(v['flight_macro_f1']) for _, v in g_],
                                 'Coarse accuracy': [ms(v['coarse_acc']) if v['coarse_acc'].notna().any() else 'n/a' for _, v in g_],
                                 'Flight ECE (10 bins)': [ms(v['flight_ece10']) if v['flight_ece10'].notna().any() else 'n/a' for _, v in g_]})
            t16b.to_csv(T / 't16b_published_architectures_flight.csv', index=False)
            md.append('## Table 16b. Published architectures at the flight level on the same balanced test sets, mean ± std over five repetitions\n\n' + md_table(t16b))
        if (R / 'e7_baselines_loso.csv').exists():
            lb = pd.read_csv(R / 'e7_baselines_loso.csv'); lb['held_out_subtype'] = lb['held_out_subtype'].map(lambda v: SUBTYPE_LABEL.get(v, v))
            keep = [c for c in ['held_out_subtype', 'n_unseen', 'n_seen_test', 'lstm_gru_unseen_acc', 'lstm_gru_seen_acc', 'lstm_gru_unseen_coarse_acc', 'lstm_gru_seen_coarse_acc',
                                'lstm_gru_unseen_conf', 'one_class_unseen_detected', 'one_class_seen_detected'] if c in lb.columns]
            t16c = lb[keep].rename(columns={'held_out_subtype': 'Held-out subtype', 'n_unseen': 'n unseen', 'n_seen_test': 'n seen test',
                                            'lstm_gru_unseen_acc': 'LSTM-GRU accuracy, unseen', 'lstm_gru_seen_acc': 'LSTM-GRU accuracy, seen',
                                            'lstm_gru_unseen_coarse_acc': 'LSTM-GRU coarse, unseen', 'lstm_gru_seen_coarse_acc': 'LSTM-GRU coarse, seen',
                                            'lstm_gru_unseen_conf': 'LSTM-GRU confidence, unseen', 'one_class_unseen_detected': 'One-class detected, unseen',
                                            'one_class_seen_detected': 'One-class detected, seen'})
            t16c.to_csv(T / 't16c_published_architectures_unseen.csv', index=False)
            md.append('## Table 16c. Published architectures on the withheld subtype and on the seen-subtype test set (single fit per hold-out)\n\n' + md_table(t16c))

    # ---------------------------------------------------------------- T8 feature importance, T9 flight-model coefficients
    if (R / 'e1_rep0_window_feature_importance.csv').exists():
        imp = pd.read_csv(R / 'e1_rep0_window_feature_importance.csv').head(15)
        imp.to_csv(T / 't8_window_feature_importance_top15.csv', index=False)
        md.append('## Table 8. Window detector, top-15 feature importance (repetition 0)\n\n' + md_table(imp, '.4f'))
        fig, ax = plt.subplots(figsize=(5.0, 3.6))
        ax.barh(imp['feature'][::-1], imp['importance'][::-1]); ax.set_xlabel('Gain importance'); ax.set_title('Window detector: top-15 features')
        save(fig, 'f_feature_importance')
    coef = pd.read_csv(R / 'e1_rep0_flight_model_coefficients.csv', index_col=0)
    coef.columns = [LABEL.get(c, c) for c in coef.columns]
    top = coef.abs().max(axis=1).sort_values(ascending=False).head(20).index
    t9 = coef.loc[top].reset_index().rename(columns={'index': 'Aggregate feature'})
    t9.to_csv(T / 't9_flight_model_coefficients_top20.csv', index=False)
    md.append('## Table 9. Flight-level model, standardised coefficients of the 20 most influential aggregates (repetition 0)\n\n' + md_table(t9))
    fig, ax = plt.subplots(figsize=(5.4, 5.2))
    im = ax.imshow(coef.loc[top].to_numpy(), aspect='auto', cmap='coolwarm', vmin=-np.abs(coef.loc[top].to_numpy()).max(), vmax=np.abs(coef.loc[top].to_numpy()).max())
    ax.set_xticks(range(coef.shape[1])); ax.set_xticklabels(coef.columns, rotation=20, ha='right'); ax.set_yticks(range(len(top))); ax.set_yticklabels(top, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.04, label='Coefficient'); ax.set_title('Flight-level model coefficients')
    save(fig, 'f_flight_model_coefficients')

    (T / 'tables.md').write_text('\n\n'.join(md) + '\n')
    print('tables ->', T, '| figures ->', FIG)
    print('\n\n'.join(md))


if __name__ == '__main__':
    main()


In [ ]:
RUN, FEAT = "v4", "features_v3"
subprocess.run(["pip", "install", "-q", "xgboost", "scikit-learn", "scipy", "pandas", "torch"], check=False)
out = P.REPORTS / RUN
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_model.py"), "--features", str(P.FEATURES / FEAT), "--out", str(out),
                         "--only", "e7,e6,e5", "--real_features", str(P.FEATURES / "features_real"), "--e7_epochs", "6"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())


In [ ]:
# ---- presentation: published architectures under the same protocol ----
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
lk = pd.read_csv(out / "e7_baselines_leakage.csv"); e0 = pd.read_csv(out / "e0_leakage_audit.csv"); e0["model"] = "xgboost"
allk = pd.concat([e0[["rep", "model", "tag", "acc", "macro_f1", "ece"]], lk[["rep", "model", "tag", "acc", "macro_f1", "ece"]]], ignore_index=True)
display(allk.groupby(["model", "tag"])[["acc", "macro_f1", "ece"]].agg(["mean", "std"]).round(3))
for m in allk.model.unique():
    piv = allk[allk.model == m].pivot(index="rep", columns="tag", values="acc")
    d = piv["random_window_split"] - piv["flight_grouped_split"]
    print(f"{m:12s} paired accuracy difference random - grouped: {d.mean():.3f} ± {d.std(ddof=0):.3f}")
display(pd.read_csv(out / "e7_baselines_flight.csv").groupby("model")[["flight_acc", "flight_macro_f1", "coarse_acc", "flight_ece10"]].agg(["mean", "std"]).round(3))
display(pd.read_csv(out / "e7_baselines_loso.csv").round(3))


In [ ]:
# ---- presentation: missingness on real flights and the 30 s onset criterion ----
e6 = pd.read_csv(out / "e6_real_summary.csv").T.rename(columns={0: "value"})
display(e6.loc[[i for i in e6.index if "missing" in i or i in ("n_flights", "gate_mahal_withheld", "abstain_fine_rate")]])
vr = pd.read_csv(out / "e6_real_flight_verdicts.csv")
display(vr[["flight_id", "pred", "conf", "abstain_fine", "gate_mahal", "missing_frac"]].sort_values("missing_frac", ascending=False).head(12).round(3))
print("correlation (missing fraction, gate flag):", round(vr["missing_frac"].corr(vr["gate_mahal"].astype(float)), 3))
e5 = pd.read_csv(out / "e5_onset_summary.csv")
display(e5[["method", "family", "subtype", "n", "detected_within_60s", "within_30s", "within_10s", "median_abs_error_s", "early_alert_rate"]].round(3))


In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = Path("/content/drive/MyDrive/UAV_GNSS_Research/uav-gnss-triage")
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive; drive.mount("/content/drive")
for f in (".gitconfig", ".git-credentials"):
    src = REPO.parent / f
    if src.exists():
        import shutil; shutil.copy(src, Path.home() / f)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)
os.chdir(REPO)
r = subprocess.run([sys.executable, "tools/commit_cell.py",
                    "v4: published architectures under the flight-level protocol (E7), real-log missingness, 30 s onset criterion; notebook 08"],
                   capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-800:])